# Color Selection for Visualizations

This notebook explores principles, techniques, and best practices for choosing effective colors in data visualizations. We'll cover color theory fundamentals, palette types, accessibility considerations, and implementation across different visualization libraries.

## Table of Contents
1. [Import Required Libraries](#import-required-libraries)
2. [Understanding Color Theory for Data Visualization](#understanding-color-theory-for-data-visualization)
3. [Color Palettes and Types](#color-palettes-and-types)
4. [Choosing Colors Based on Data Types](#choosing-colors-based-on-data-types)
5. [Color Maps in Matplotlib](#color-maps-in-matplotlib)
6. [Custom Color Palettes with Seaborn](#custom-color-palettes-with-seaborn)
7. [Colorblind-friendly Visualizations](#colorblind-friendly-visualizations)
8. [Using Color to Highlight Information](#using-color-to-highlight-information)
9. [Interactive Color Selection with Plotly](#interactive-color-selection-with-plotly)
10. [Best Practices for Color Usage](#best-practices-for-color-usage)
11. [Color and Chart Types](#color-and-chart-types)

## Import Required Libraries
Let's import the necessary visualization libraries including matplotlib, seaborn, plotly, and additional color-related modules.

In [ ]:
# Import core visualization libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib import cm
import colorcet as cc  # Additional colormaps
from colorspacious import cspace_converter

# For color conversion and manipulation
from matplotlib import colors
import matplotlib.colors as mcolors

# For image processing (to extract colors from images)
from PIL import Image
from io import BytesIO
import requests

# Configure visualization settings
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

## Understanding Color Theory for Data Visualization

Color theory provides the foundation for effective data visualization. Understanding how colors work together and how they are perceived is crucial for creating clear and impactful visualizations.

### Basic Color Theory Concepts

* **Primary Colors**: Red, Yellow, Blue - cannot be created by mixing other colors
* **Secondary Colors**: Green, Orange, Purple - created by mixing primary colors
* **Tertiary Colors**: Created by mixing primary and secondary colors
* **Color Properties**:
  - **Hue**: The actual color (red, blue, etc.)
  - **Saturation**: The intensity or purity of the color
  - **Value/Brightness**: The lightness or darkness of a color

### Color Harmony and Relationships

* **Complementary colors**: Colors opposite each other on the color wheel
* **Analogous colors**: Colors adjacent to each other on the color wheel
* **Triadic colors**: Three colors equally spaced on the color wheel

### Psychological Impact of Colors

Different colors evoke different psychological responses and can affect how data is interpreted.

In [ ]:
# Create a visualization of the color wheel to explain color relationships
def create_color_wheel(size=400, with_labels=True):
    """Create a color wheel visualization"""
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
    
    # Generate hues around the color wheel
    theta = np.linspace(0, 2*np.pi, 100)
    radii = np.ones_like(theta)
    
    # Create a colormap for the wheel
    cmap = LinearSegmentedColormap.from_list(
        'hsv', 
        [(np.cos(t), np.sin(t), 0.5) for t in np.linspace(0, 2*np.pi, 100)],
        N=100
    )
    
    # Plot the wheel
    sc = ax.scatter(theta, radii, c=theta, s=300, cmap=cmap, alpha=0.8)
    
    # Add color relationship labels
    if with_labels:
        # Primary colors
        ax.text(0, 1.25, "Red (Primary)", ha='center', fontweight='bold', color='red')
        ax.text(2*np.pi/3, 1.25, "Blue (Primary)", ha='center', fontweight='bold', color='blue')
        ax.text(4*np.pi/3, 1.25, "Yellow (Primary)", ha='center', fontweight='bold', color='goldenrod')
        
        # Secondary colors
        ax.text(np.pi/3, 1.15, "Orange (Secondary)", ha='center', fontweight='bold', color='orange')
        ax.text(3*np.pi/3, 1.15, "Green (Secondary)", ha='center', fontweight='bold', color='green')
        ax.text(5*np.pi/3, 1.15, "Purple (Secondary)", ha='center', fontweight='bold', color='purple')
    
    # Remove the radial grid, ticks, and labels
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines['polar'].set_visible(False)
    
    plt.title('Color Wheel Showing Color Relationships', fontsize=16)
    return fig, ax

create_color_wheel()
plt.show()

### Color Perception and Data Interpretation

How we perceive colors affects how we interpret data visualizations:

1. **Color Contrast** affects readability and visual hierarchy
2. **Color Associations** carry cultural and contextual meanings  
3. **Perceptual Uniformity** ensures color changes are perceived proportionally to data changes

Let's look at how the same data can be interpreted differently with different color choices:

In [ ]:
# Create sample data
np.random.seed(42)
data = np.random.normal(0, 1, 100).cumsum()

# Create the same line chart with different color schemes to show impact
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Neutral color - factual interpretation
axs[0].plot(data, color='#4C72B0', linewidth=2.5)  # Blue
axs[0].set_title('Neutral Color (Blue)\nSuggests factual information', fontsize=12)
axs[0].grid(True, alpha=0.3)

# Red - negative/alarming interpretation
axs[1].plot(data, color='#C44E52', linewidth=2.5)  # Red
axs[1].set_title('Red Color\nCan suggest negative/alarming trend', fontsize=12)
axs[1].grid(True, alpha=0.3)

# Green - positive interpretation
axs[2].plot(data, color='#55A868', linewidth=2.5)  # Green
axs[2].set_title('Green Color\nCan suggest positive/favorable trend', fontsize=12)
axs[2].grid(True, alpha=0.3)

for ax in axs:
    ax.set_ylim(data.min()-1, data.max()+1)
    ax.set_xlabel('Time')
    ax.set_ylabel('Value')

plt.tight_layout()
plt.suptitle('How Color Choice Affects Data Interpretation', fontsize=16, y=1.05)
plt.show()

## Color Palettes and Types

When selecting colors for data visualizations, you typically need to choose from one of three main types of color palettes:

### 1. Sequential Palettes
- Used for data that progresses from low to high values
- Typically vary in lightness and saturation of a single hue or multiple hues
- Best for showing continuous data like temperature, population density

### 2. Diverging Palettes
- Used for data with meaningful mid-point (zero, average)
- Typically has two different hues that diverge from a neutral central color
- Best for data that deviates from a central value, like profit/loss, temperature anomalies

### 3. Qualitative Palettes
- Used for nominal/categorical data with no inherent order
- Consists of different hues with similar lightness/saturation
- Best for distinguishing discrete categories like product types, regions, political parties

Let's visualize examples of each palette type:

In [ ]:
# Create function to display color palettes
def display_color_palettes():
    # Create figure with subplots
    fig, axs = plt.subplots(3, 1, figsize=(10, 8))
    
    # 1. Sequential palettes
    sequential_palettes = ['Blues', 'Greens', 'Reds', 'YlOrBr', 'viridis', 'plasma']
    sequential_data = []
    
    for pal in sequential_palettes:
        # Get colors from colormap
        if pal in plt.colormaps():
            cmap = plt.get_cmap(pal)
            colors = [cmap(i/10) for i in range(11)]
            sequential_data.append((pal, colors))
    
    # 2. Diverging palettes
    diverging_palettes = ['RdBu_r', 'PiYG', 'PRGn', 'BrBG', 'RdYlGn', 'coolwarm']
    diverging_data = []
    
    for pal in diverging_palettes:
        if pal in plt.colormaps():
            cmap = plt.get_cmap(pal)
            colors = [cmap(i/10) for i in range(11)]
            diverging_data.append((pal, colors))
    
    # 3. Qualitative palettes
    qualitative_palettes = ['Set1', 'Set2', 'Set3', 'Dark2', 'tab10', 'Pastel1']
    qualitative_data = []
    
    for pal in qualitative_palettes:
        if pal in plt.colormaps():
            cmap = plt.get_cmap(pal)
            # Get only as many colors as the colormap has
            n_colors = min(10, cmap.N)
            colors = [cmap(i) for i in range(n_colors)]
            # Fill with white if less than 10 colors
            colors.extend([(1, 1, 1, 1)] * (10 - len(colors)))
            qualitative_data.append((pal, colors))
    
    # Plot each palette type
    titles = ["Sequential Palettes", "Diverging Palettes", "Qualitative Palettes"]
    palette_data = [sequential_data, diverging_data, qualitative_data]
    
    for ax, title, data in zip(axs, titles, palette_data):
        ax.set_title(title, fontsize=14)
        for i, (name, colors) in enumerate(data):
            for j, color in enumerate(colors):
                ax.fill_between([j, j+1], [i, i], [i+0.9, i+0.9], color=color)
            ax.text(-0.5, i+0.45, name, ha='right', va='center', fontsize=10)
        
        ax.set_xlim(0, 11)
        ax.set_ylim(-0.1, len(data)-0.1)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_frame_on(False)
    
    plt.tight_layout()
    plt.suptitle('Types of Color Palettes for Data Visualization', fontsize=16, y=0.98)
    plt.show()

display_color_palettes()

### When to Use Each Palette Type

Let's demonstrate each palette type with appropriate data examples:

In [ ]:
# # Generate sample data for each palette type
# np.random.seed(42)

# # Create sample data for each palette type
# # 1. Sequential data (population density)
# states = ['State ' + str(i) for i in range(1, 11)]
# population_density = np.sort(np.random.randint(10, 500, size=10))
# seq_data = pd.DataFrame({'State': states, 'Population Density': population_density})

# # 2. Diverging data (temperature anomalies)
# months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
# temp_anomalies = np.random.normal(0, 2, 12)
# div_data = pd.DataFrame({'Month': months, 'Temperature Anomaly (°C)': temp_anomalies})

# # 3. Qualitative data (sales by category)
# categories = ['Electronics', 'Clothing', 'Food', 'Books', 'Home', 'Sports']
# sales = np.random.randint(100, 1000, size=6)
# qual_data = pd.DataFrame({'Category': categories, 'Sales': sales})

# # Create visualizations with appropriate color palettes
# fig, axs = plt.subplots(3, 1, figsize=(14, 12))

# # 1. Sequential palette example
# sns.barplot(x='State', y='Population Density', data=seq_data, ax=axs[0], palette='viridis')
# axs[0].set_title('Sequential Palette: Population Density by State', fontsize=14)
# axs[0].set_ylabel('Population Density (people/km²)')
# axs[0].tick_params(axis='x', rotation=45)

# # 2. Diverging palette example
# div_palette = sns.color_palette("coolwarm", n_colors=len(months))
# bars = axs[1].bar(div_data['Month'], div_data['Temperature Anomaly (°C)'], color=div_palette)
# axs[1].set_title('Diverging Palette: Monthly Temperature Anomalies', fontsize=14)
# axs[1].set_ylabel('Temperature Anomaly (°C)')
# axs[1].axhline(y=0, color='black', linestyle='-', alpha=0.2)

# # 3. Qualitative palette example
# sns.barplot(x='Category', y='Sales', data=qual_data, ax=axs[2], palette='Set2')
# axs[2].set_title('Qualitative Palette: Sales by Product Category', fontsize=14)
# axs[2].set_ylabel('Sales (thousands)')
# axs[2].tick_params(axis='x', rotation=45)

# plt.tight_layout()
# plt.show()

```
FutureWarning: 
Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.
```

In [ ]:
# Generate sample data for each palette type
np.random.seed(42)

# Create sample data for each palette type
# 1. Sequential data (population density)
states = ['State ' + str(i) for i in range(1, 11)]
population_density = np.sort(np.random.randint(10, 500, size=10))
seq_data = pd.DataFrame({'State': states, 'Population Density': population_density})

# 2. Diverging data (temperature anomalies)
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
temp_anomalies = np.random.normal(0, 2, 12)
div_data = pd.DataFrame({'Month': months, 'Temperature Anomaly (°C)': temp_anomalies})

# 3. Qualitative data (sales by category)
categories = ['Electronics', 'Clothing', 'Food', 'Books', 'Home', 'Sports']
sales = np.random.randint(100, 1000, size=6)
qual_data = pd.DataFrame({'Category': categories, 'Sales': sales})

# Create visualizations with appropriate color palettes
fig, axs = plt.subplots(3, 1, figsize=(14, 12))

# 1. Sequential palette example
sns.barplot(x='State', y='Population Density', data=seq_data, ax=axs[0], palette='viridis', hue='State')
axs[0].set_title('Sequential Palette: Population Density by State', fontsize=14)
axs[0].set_ylabel('Population Density (people/km²)')
axs[0].tick_params(axis='x', rotation=45)

# 2. Diverging palette example
div_palette = sns.color_palette("coolwarm", n_colors=len(months))
bars = axs[1].bar(div_data['Month'], div_data['Temperature Anomaly (°C)'], color=div_palette)
axs[1].set_title('Diverging Palette: Monthly Temperature Anomalies', fontsize=14)
axs[1].set_ylabel('Temperature Anomaly (°C)')
axs[1].axhline(y=0, color='black', linestyle='-', alpha=0.2)

# 3. Qualitative palette example
sns.barplot(x='Category', y='Sales', data=qual_data, ax=axs[2], palette='Set2', hue='Category')
axs[2].set_title('Qualitative Palette: Sales by Product Category', fontsize=14)
axs[2].set_ylabel('Sales (thousands)')
axs[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Choosing Colors Based on Data Types

Different types of data require different color approaches. Here's a guide for choosing colors based on data types:

### 1. For Categorical/Nominal Data
- Use distinctly different hues with similar brightness and saturation
- Focus on distinguishable colors rather than related ones
- Consider the number of categories (more categories = more challenging)

### 2. For Ordinal Data
- Use variations of the same hue or related hues
- Vary saturation or value to show order
- Ensure that the order is visually apparent

### 3. For Quantitative/Continuous Data
- Sequential palettes for unidirectional data
- Diverging palettes for bidirectional data
- Ensure perceptual uniformity for accurate interpretation

### 4. For Different Chart Types
- Area charts/heat maps: Sequential palettes work well
- Scatter plots: Consider color intensity for density or a third variable
- Multi-series line charts: Qualitative palettes to distinguish series

Let's demonstrate these principles with examples:

In [ ]:
# Create subplots for different data and chart types
fig = plt.figure(figsize=(16, 12))

# 1. Categorical data - bar chart with qualitative palette
plt.subplot(2, 2, 1)
categories = ['Group A', 'Group B', 'Group C', 'Group D', 'Group E']
values = np.random.randint(10, 100, size=5)
plt.bar(categories, values, color=sns.color_palette("Set2", 5))
plt.title('Categorical Data - Qualitative Palette', fontsize=12)
plt.ylabel('Value')

# 2. Ordinal data - bar chart with sequential single-hue palette
plt.subplot(2, 2, 2)
ratings = ['Very Poor', 'Poor', 'Average', 'Good', 'Very Good']
scores = [15, 25, 40, 65, 85]
plt.bar(ratings, scores, color=sns.color_palette("Blues", 5))
plt.title('Ordinal Data - Sequential Single-Hue Palette', fontsize=12)
plt.ylabel('Score')

# 3. Quantitative data - heatmap with sequential palette
plt.subplot(2, 2, 3)
np.random.seed(42)
data = np.random.rand(10, 10)
sns.heatmap(data, cmap="YlGnBu", annot=False, cbar=True)
plt.title('Quantitative Data - Heatmap with Sequential Palette', fontsize=12)

# 4. Diverging data - heatmap with diverging palette
plt.subplot(2, 2, 4)
np.random.seed(42)
corr_data = np.random.uniform(-1, 1, size=(8, 8))
# Make it symmetric like a correlation matrix
corr_data = (corr_data + corr_data.T) / 2
np.fill_diagonal(corr_data, 1)
sns.heatmap(corr_data, cmap="coolwarm", annot=False, cbar=True, vmin=-1, vmax=1)
plt.title('Diverging Data - Correlation Matrix with Diverging Palette', fontsize=12)

plt.tight_layout()
plt.suptitle('Choosing Colors Based on Data Types', fontsize=16, y=1.02)
plt.show()

### Considerations for Multiple Variables

When visualizing multiple variables, carefully consider how color can help distinguish between them:

In [ ]:
# Create a scatterplot with multiple variables encoded using color and size
np.random.seed(42)
n = 50
df = pd.DataFrame({
    'x': np.random.rand(n),
    'y': np.random.rand(n),
    'category': np.random.choice(['Category A', 'Category B', 'Category C'], n),
    'value': np.random.randint(10, 100, n)
})

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
# Using color for categories (qualitative) and size for quantitative value
scatter = plt.scatter(df['x'], df['y'], 
                     c=[{'Category A': 0, 'Category B': 1, 'Category C': 2}[cat] for cat in df['category']],
                     cmap='Set1', 
                     s=df['value'] * 2,  # Size represents value
                     alpha=0.7)

# Create color legend
unique_categories = df['category'].unique()
handles = [plt.Line2D([0], [0], marker='o', color='w', 
                     markerfacecolor=plt.cm.Set1(i/3), markersize=10) 
          for i in range(len(unique_categories))]
plt.legend(handles, unique_categories, title='Category', loc='upper right')

plt.title('Multi-variable Visualization:\nColor for Categories, Size for Value', fontsize=12)
plt.xlabel('X Variable')
plt.ylabel('Y Variable')

plt.subplot(1, 2, 2)
# Using a continuous colormap for the value
scatter = plt.scatter(df['x'], df['y'], 
                     c=df['value'],
                     cmap='viridis', 
                     s=80,
                     alpha=0.7)

plt.colorbar(scatter, label='Value')
plt.title('Using Color to Encode a Continuous Variable', fontsize=12)
plt.xlabel('X Variable')
plt.ylabel('Y Variable')

plt.tight_layout()
plt.show()

## Color Maps in Matplotlib

Matplotlib provides a wide range of colormaps to visualize different types of data. Understanding which colormap to use is crucial for effective data visualization.

### Types of Matplotlib Colormaps:

1. **Sequential**: Single color progression from low to high values
2. **Diverging**: Two different colors diverging from a central neutral color
3. **Cyclic**: Colors that wrap around to show periodic data
4. **Qualitative**: Distinct colors for categorical data

### Perceptually Uniform Colormaps

Perceptually uniform colormaps (like `viridis`, `plasma`, `inferno`, `magma`) maintain consistent perceived color differences throughout the range of values. This ensures that equal steps in data are represented by equal perceptual steps in color.

Let's explore various matplotlib colormaps:

In [ ]:
# Define function to display multiple colormaps
def display_colormaps(cmap_category, cmap_list):
    """Display a set of colormaps from matplotlib"""
    fig, axes = plt.subplots(len(cmap_list), 1, figsize=(10, 0.5 * len(cmap_list)))
    
    if len(cmap_list) == 1:
        axes = [axes]  # Make it iterable when there's only one colormap
    
    gradient = np.linspace(0, 1, 256)
    gradient = np.vstack((gradient, gradient))
    
    for ax, cmap_name in zip(axes, cmap_list):
        cmap = plt.get_cmap(cmap_name)
        ax.imshow(gradient, aspect='auto', cmap=cmap)
        ax.text(-0.01, 0.5, cmap_name, va='center', ha='right', fontsize=10, transform=ax.transAxes)
        ax.set_xticks([])
        ax.set_yticks([])
    
    plt.tight_layout()
    plt.suptitle(f'{cmap_category} Colormaps in Matplotlib', fontsize=14, y=1.01)
    plt.show()

# Show different categories of colormaps
perceptually_uniform = ['viridis', 'plasma', 'inferno', 'magma', 'cividis']
display_colormaps('Perceptually Uniform', perceptually_uniform)

sequential = ['Blues', 'BuGn', 'BuPu', 'GnBu', 'Greens', 'Greys', 'OrRd', 'Oranges', 'PuBu', 'PuRd', 'Purples', 'RdPu', 'Reds', 'YlGn', 'YlGnBu', 'YlOrBr', 'YlOrRd']
display_colormaps('Sequential', sequential[:10])  # Show first 10 for brevity

diverging = ['BrBG', 'PRGn', 'PiYG', 'PuOr', 'RdBu', 'RdGy', 'RdYlBu', 'RdYlGn', 'Spectral', 'coolwarm', 'bwr', 'seismic']
display_colormaps('Diverging', diverging[:8])  # Show first 8 for brevity

qualitative = ['Pastel1', 'Pastel2', 'Paired', 'Accent', 'Dark2', 'Set1', 'Set2', 'Set3', 'tab10', 'tab20', 'tab20b', 'tab20c']
display_colormaps('Qualitative', qualitative[:8])  # Show first 8 for brevity

cyclic = ['twilight', 'twilight_shifted', 'hsv']
display_colormaps('Cyclic', cyclic)

### Applying Colormaps to Visualizations

Let's demonstrate how to apply these colormaps to different visualization types:

In [ ]:
# Create sample data
np.random.seed(42)
x = np.linspace(0, 5, 100)
y = np.linspace(0, 5, 100)
X, Y = np.meshgrid(x, y)
Z1 = np.sin(X) * np.cos(Y)  # For contour plot
Z2 = (X - 2.5) ** 2 + (Y - 2.5) ** 2  # For heatmap (distance from center)
random_data = np.random.random((5, 5))  # Random data for heatmap

fig, axs = plt.subplots(2, 2, figsize=(14, 10))

# 1. Contour plot with viridis colormap
contour = axs[0, 0].contourf(X, Y, Z1, 20, cmap='viridis')
axs[0, 0].set_title('Contour Plot with Viridis (Perceptually Uniform)')
axs[0, 0].set_xlabel('X')
axs[0, 0].set_ylabel('Y')
plt.colorbar(contour, ax=axs[0, 0])

# 2. Heatmap with coolwarm (diverging)
im = axs[0, 1].imshow(Z1, cmap='coolwarm', origin='lower', extent=[0, 5, 0, 5])
axs[0, 1].set_title('Heatmap with Coolwarm (Diverging)')
axs[0, 1].set_xlabel('X')
axs[0, 1].set_ylabel('Y')
plt.colorbar(im, ax=axs[0, 1])

# 3. Scatter plot with plasma colormap
points = axs[1, 0].scatter(
    np.random.random(50), 
    np.random.random(50), 
    c=np.random.random(50), 
    cmap='plasma', 
    s=100
)
axs[1, 0].set_title('Scatter Plot with Plasma Colormap')
axs[1, 0].set_xlabel('X')
axs[1, 0].set_ylabel('Y')
plt.colorbar(points, ax=axs[1, 0], label='Value')

# 4. 3D Surface plot with YlGnBu colormap
from mpl_toolkits.mplot3d import Axes3D
axs[1, 1].remove()  # Remove the existing axes
ax = fig.add_subplot(2, 2, 4, projection='3d')
surface = ax.plot_surface(X, Y, Z1, cmap='YlGnBu', linewidth=0, antialiased=False)
ax.set_title('3D Surface with YlGnBu Colormap')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
plt.colorbar(surface, ax=ax, shrink=0.5, aspect=5)

plt.tight_layout()
plt.show()

### Choosing the Right Colormap

Here's how to choose the appropriate colormap:

- **For continuous data without a critical midpoint**: Sequential colormaps like `viridis`, `plasma`, or `Blues`
- **For diverging data with a meaningful midpoint**: Diverging colormaps like `coolwarm`, `RdBu`, or `BrBG`
- **For cyclic/periodic data**: Cyclic colormaps like `twilight` or `hsv`
- **For categorical data**: Qualitative colormaps like `Set1`, `tab10`, or `Paired`

Let's compare some colormap examples for the same data:

In [ ]:
# Generate sample data for comparison
np.random.seed(42)
data = np.random.randn(20, 20)  # Random data
data_smooth = np.zeros((20, 20))  # Smoothed gradient data
for i in range(20):
    for j in range(20):
        data_smooth[i, j] = np.sin(i/5) * np.cos(j/5)

# Function to compare colormaps
def compare_colormaps(data, cmaps, title):
    """Compare multiple colormaps on the same data"""
    fig, axes = plt.subplots(1, len(cmaps), figsize=(15, 4))
    
    if len(cmaps) == 1:
        axes = [axes]
    
    for ax, cmap in zip(axes, cmaps):
        im = ax.imshow(data, cmap=cmap)
        ax.set_title(cmap)
        ax.set_xticks([])
        ax.set_yticks([])
        plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05)
    
    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

# Compare sequential colormaps
sequential_cmaps = ['viridis', 'plasma', 'Blues', 'YlGnBu']
compare_colormaps(data_smooth, sequential_cmaps, 'Sequential Colormaps Comparison')

# Compare diverging colormaps
diverging_cmaps = ['coolwarm', 'RdBu', 'seismic', 'PiYG']
compare_colormaps(data_smooth, diverging_cmaps, 'Diverging Colormaps Comparison')

# Compare perceptual uniformity
# Demonstrate why rainbow maps like 'jet' are problematic
problematic_vs_good = ['jet', 'rainbow', 'nipy_spectral', 'viridis']
compare_colormaps(data_smooth, problematic_vs_good, 'Problematic vs. Perceptually Uniform Colormaps')

## Custom Color Palettes with Seaborn

Seaborn makes it easy to create custom color palettes that are visually appealing and suited to your specific data visualization needs. Let's explore how to create and use custom color palettes in Seaborn.

### Types of Seaborn Color Palettes

1. **Built-in named palettes**: `'deep'`, `'muted'`, `'pastel'`, `'bright'`, `'dark'`, and `'colorblind'`
2. **Color brewer palettes**: Access using `sns.color_palette("Blues", 8)` syntax
3. **Custom palettes**: Created from individual colors or by extracting colors from images

Let's explore these options:

In [ ]:
# # Display Seaborn's built-in color palettes
# def show_seaborn_palettes():
#     """Display built-in Seaborn palettes"""
#     # List of seaborn palette names
#     palette_names = ['deep', 'muted', 'pastel', 'bright', 'dark', 'colorblind']
    
#     # Create a figure with subplots
#     fig, axes = plt.subplots(len(palette_names), 1, figsize=(10, len(palette_names)))
    
#     # Plot each palette
#     for i, palette_name in enumerate(palette_names):
#         # Get the color palette
#         current_palette = sns.color_palette(palette_name)
#         # Plot color swatches
#         sns.palplot(current_palette, ax=axes[i])
#         # Add the palette name as label
#         axes[i].text(-0.1, 0.5, palette_name, transform=axes[i].transAxes, 
#                     ha='right', va='center', fontsize=12)
#         # Remove x and y ticks
#         axes[i].set_xticks([])
#         axes[i].set_yticks([])
    
#     plt.suptitle('Seaborn Built-in Color Palettes', fontsize=16, y=1.02)
#     plt.tight_layout()
#     plt.show()

# show_seaborn_palettes()

# # Create and visualize custom color palettes
# plt.figure(figsize=(10, 6))

# # 1. Custom categorical palette from individual colors
# plt.subplot(3, 1, 1)
# custom_palette1 = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854']
# sns.palplot(custom_palette1)
# plt.title('Custom Palette from Individual Colors', fontsize=12)

# # 2. Sequential palette using seaborn's color_palette
# plt.subplot(3, 1, 2)
# custom_palette2 = sns.color_palette("Blues_r", n_colors=7)
# sns.palplot(custom_palette2)
# plt.title('Custom Sequential Blue Palette', fontsize=12)

# # 3. Diverging palette with custom center
# plt.subplot(3, 1, 3)
# custom_palette3 = sns.diverging_palette(250, 15, s=75, l=60, n=9, center="dark")
# sns.palplot(custom_palette3)
# plt.title('Custom Diverging Palette', fontsize=12)

# plt.tight_layout()
# plt.suptitle('Creating Custom Color Palettes in Seaborn', fontsize=14, y=1.02)
# plt.show()

```
TypeError: palplot() got an unexpected keyword argument 'ax'
```

In [ ]:
# Function to convert hex color codes to RGB (0-1 range)
def hex_to_rgb(hex_list):
    rgb_list = []
    for hex_color in hex_list:
        # Remove the '#' and convert hex to RGB
        rgb = [int(hex_color[i:i+2], 16) / 255.0 for i in (1, 3, 5)]  # Normalize to 0-1 range
        rgb_list.append(rgb)
    return np.array(rgb_list)

# Display Seaborn's built-in color palettes
def show_seaborn_palettes():
    """Display built-in Seaborn palettes"""
    # List of seaborn palette names
    palette_names = ['deep', 'muted', 'pastel', 'bright', 'dark', 'colorblind']
    
    # Create a figure with subplots
    fig, axes = plt.subplots(len(palette_names), 1, figsize=(10, len(palette_names)))
    
    # Plot each palette
    for i, palette_name in enumerate(palette_names):
        # Get the color palette (this returns a list of RGB tuples)
        current_palette = sns.color_palette(palette_name)
        # Plot color swatches using imshow
        axes[i].imshow([current_palette], aspect='auto')
        # Add the palette name as label
        axes[i].text(-0.1, 0.5, palette_name, transform=axes[i].transAxes, 
                    ha='right', va='center', fontsize=12)
        # Remove x and y ticks
        axes[i].set_xticks([])
        axes[i].set_yticks([])
    
    plt.suptitle('Seaborn Built-in Color Palettes', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

show_seaborn_palettes()

# Create and visualize custom color palettes
plt.figure(figsize=(10, 6))

# 1. Custom categorical palette from individual colors
plt.subplot(3, 1, 1)
custom_palette1 = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854']
custom_palette1_rgb = hex_to_rgb(custom_palette1)  # Convert hex to RGB
plt.imshow([custom_palette1_rgb], aspect='auto')
plt.title('Custom Palette from Individual Colors', fontsize=12)
plt.axis('off')

# 2. Sequential palette using seaborn's color_palette
plt.subplot(3, 1, 2)
custom_palette2 = sns.color_palette("Blues_r", n_colors=7)
plt.imshow([custom_palette2], aspect='auto')
plt.title('Custom Sequential Blue Palette', fontsize=12)
plt.axis('off')

# 3. Diverging palette with custom center
plt.subplot(3, 1, 3)
custom_palette3 = sns.diverging_palette(250, 15, s=75, l=60, n=9, center="dark")
plt.imshow([custom_palette3], aspect='auto')
plt.title('Custom Diverging Palette', fontsize=12)
plt.axis('off')

plt.tight_layout()
plt.suptitle('Creating Custom Color Palettes in Seaborn', fontsize=14, y=1.02)
plt.show()

### Extracting Color Palettes from Images

A creative approach to choosing colors is extracting palettes from images. This can help create visually harmonious visualizations or match a specific brand's visual identity.

In [ ]:
# Function to extract a color palette from an image
def extract_colors_from_image(image_url, n_colors=5):
    """Extract dominant colors from an image using k-means clustering"""
    try:
        # Download the image
        response = requests.get(image_url)
        image = Image.open(BytesIO(response.content))
        
        # Resize image to speed up processing
        image = image.resize((100, 100))
        
        # Convert image to RGB array
        image_array = np.array(image)
        
        # Reshape to list of pixels
        pixels = image_array.reshape(-1, 3)
        
        # Perform k-means clustering to find dominant colors
        from sklearn.cluster import KMeans
        kmeans = KMeans(n_clusters=n_colors, random_state=42)
        kmeans.fit(pixels)
        
        # Get the colors as RGB values
        colors = kmeans.cluster_centers_.astype(int)
        
        # Convert to hex values
        hex_colors = ['#%02x%02x%02x' % tuple(color) for color in colors]
        
        # Show the image and extracted colors
        plt.figure(figsize=(10, 4))
        
        # Display the image
        plt.subplot(1, 2, 1)
        plt.imshow(image)
        plt.title('Source Image')
        plt.axis('off')
        
        # Display the color palette
        plt.subplot(1, 2, 2)
        for i, color in enumerate(hex_colors):
            plt.fill_between([0, 1], [i, i], [i+1, i+1], color=color)
        plt.xlim(0, 1)
        plt.ylim(0, n_colors)
        plt.title('Extracted Color Palette')
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        return hex_colors
        
    except Exception as e:
        print(f"Error: {e}")
        return None

# Example image URLs - nature scene with vibrant colors
image_url = "https://images.unsplash.com/photo-1506744038136-46273834b3fb?ixlib=rb-1.2.1&q=80&fm=jpg&crop=entropy&cs=tinysrgb&w=1000&fit=max"

# Extract colors from the image
palette = extract_colors_from_image(image_url, n_colors=6)
print("Extracted hex colors:", palette)

# Create a chart using the extracted palette
if palette:
    plt.figure(figsize=(10, 5))
    
    # Generate sample data
    np.random.seed(42)
    categories = ['Category A', 'Category B', 'Category C', 'Category D', 'Category E', 'Category F']
    values = np.random.randint(10, 100, size=6)
    
    # Create bar chart with the extracted palette
    plt.bar(categories, values, color=palette)
    plt.title('Bar Chart Using Image-Extracted Color Palette', fontsize=14)
    plt.ylabel('Values')
    
    plt.show()

### Applying Custom Palettes to Various Seaborn Plots

Let's apply custom color palettes to different Seaborn visualization types:

In [ ]:
# # Generate some sample data for visualization
# np.random.seed(42)
# tips = sns.load_dataset('tips')  # Load the built-in tips dataset
# iris = sns.load_dataset('iris')  # Load the built-in iris dataset

# # Create various plots with custom color palettes
# fig = plt.figure(figsize=(15, 12))

# # 1. Bar plot with custom palette
# plt.subplot(2, 2, 1)
# sns.barplot(x='day', y='total_bill', data=tips, 
#            palette=sns.color_palette("Blues_d", n_colors=4))
# plt.title('Bar Plot with Blues_d Palette', fontsize=12)

# # 2. Box plot with custom diverging palette
# plt.subplot(2, 2, 2)
# custom_palette = sns.diverging_palette(220, 20, as_cmap=False, n=4)
# sns.boxplot(x='day', y='total_bill', data=tips, palette=custom_palette)
# plt.title('Box Plot with Custom Diverging Palette', fontsize=12)

# # 3. Violin plot with cubehelix palette
# plt.subplot(2, 2, 3)
# sns.violinplot(x='day', y='total_bill', data=tips, 
#               palette=sns.cubehelix_palette(4, start=.5, rot=-.75))
# plt.title('Violin Plot with Cubehelix Palette', fontsize=12)

# # 4. Scatterplot with custom sequential palette
# plt.subplot(2, 2, 4)
# if palette:  # Use the image-extracted palette if available
#     sns.scatterplot(x='sepal_length', y='sepal_width', hue='species', 
#                    data=iris, palette=palette[:3])
#     plt.title('Scatter Plot with Image-Extracted Palette', fontsize=12)
# else:
#     # Fallback to a standard palette
#     sns.scatterplot(x='sepal_length', y='sepal_width', hue='species', 
#                    data=iris, palette='Set2')
#     plt.title('Scatter Plot with Set2 Palette', fontsize=12)

# plt.tight_layout()
# plt.suptitle('Applying Custom Color Palettes to Seaborn Plots', fontsize=16, y=1.02)
# plt.show()

```
FutureWarning: 
Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

NameError: name 'palette' is not defined
```

In [ ]:
# Sample data
np.random.seed(42)
tips  = sns.load_dataset('tips')
iris  = sns.load_dataset('iris')

fig = plt.figure(figsize=(15, 12))

# 1. Bar plot — map day to hue, suppress legend
plt.subplot(2, 2, 1)
sns.barplot(
    x='day', y='total_bill', hue='day', legend=False,
    data=tips, palette=sns.color_palette("Blues_d", n_colors=4)
)
plt.title('Bar Plot with Blues_d Palette', fontsize=12)

# 2. Box plot — same trick
plt.subplot(2, 2, 2)
custom_palette = sns.diverging_palette(220, 20, n=4)
sns.boxplot(
    x='day', y='total_bill', hue='day', legend=False,
    data=tips, palette=custom_palette
)
plt.title('Box Plot with Custom Diverging Palette', fontsize=12)

# 3. Violin plot — same trick
plt.subplot(2, 2, 3)
sns.violinplot(
    x='day', y='total_bill', hue='day', legend=False,
    data=tips, palette=sns.cubehelix_palette(4, start=.5, rot=-.75)
)
plt.title('Violin Plot with Cubehelix Palette', fontsize=12)

# 4. Scatter plot — use variable named image_palette if you created one earlier
plt.subplot(2, 2, 4)
try:
    image_palette        # does it exist?
    active_palette = image_palette[:3]
    title_text     = 'Scatter Plot with Image‑Extracted Palette'
except NameError:
    active_palette = 'Set2'
    title_text     = 'Scatter Plot with Set2 Palette'

sns.scatterplot(
    x='sepal_length', y='sepal_width', hue='species',
    data=iris, palette=active_palette
)
plt.title(title_text, fontsize=12)
plt.legend(title='')          # tidy legend title if it reappears

plt.tight_layout()
plt.suptitle('Applying Custom Color Palettes to Seaborn Plots',
             fontsize=16, y=1.02)
plt.show()

## Colorblind-friendly Visualizations

Approximately 8% of men and 0.5% of women have some form of color vision deficiency (CVD). Creating colorblind-friendly visualizations ensures your data is accessible to a wider audience.

### Types of Color Vision Deficiencies

1. **Deuteranomaly**: Reduced sensitivity to green light (most common)
2. **Protanomaly**: Reduced sensitivity to red light
3. **Tritanomaly**: Reduced sensitivity to blue light (rare)

### Best Practices for Colorblind-friendly Visualizations

1. Use highly distinguishable colors
2. Don't rely solely on color - add patterns, labels, or other visual cues
3. Use specific colorblind-friendly palettes
4. Test your visualizations with color blindness simulators

Let's create and test colorblind-friendly visualizations:

In [ ]:
# # Define a colorblind-friendly palette
# cb_palette = ['#0072B2', '#009E73', '#D55E00', '#CC79A7', '#F0E442', '#56B4E9']

# # Create sample data
# categories = ['Category A', 'Category B', 'Category C', 'Category D', 'Category E']
# values = [25, 40, 30, 55, 15]

# # Function to simulate color blindness
# def simulate_colorblindness(rgb, cvd_type):
#     """
#     Simulate color blindness for RGB values
    
#     Parameters:
#     rgb: RGB color values (0-1)
#     cvd_type: Type of color vision deficiency ('deuteranomaly', 'protanomaly', or 'tritanomaly')
    
#     Returns:
#     RGB values as they would appear to someone with the specified color vision deficiency
#     """
#     cvd_space = {"deuteranomaly": "sRGB1+D65:deuteranomaly",
#                  "protanomaly": "sRGB1+D65:protanomaly",
#                  "tritanomaly": "sRGB1+D65:tritanomaly"}
    
#     # Convert to colorblind perception
#     if cvd_type in cvd_space:
#         return cspace_converter(cvd_space[cvd_type], "sRGB1+D65")(rgb)
#     else:
#         return rgb

# # Function to create and compare visualizations under different color vision conditions
# def compare_cvd_perception(palette, chart_type='bar'):
#     """Create charts and show how they look under different types of color blindness"""
#     cvd_types = ['original', 'deuteranomaly', 'protanomaly', 'tritanomaly']
    
#     fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    
#     for i, cvd in enumerate(cvd_types):
#         if chart_type == 'bar':
#             if cvd == 'original':
#                 # Original colors
#                 axes[i].bar(categories, values, color=palette[:len(categories)])
#             else:
#                 # Apply CVD simulation to each color
#                 cvd_colors = []
#                 for color in palette[:len(categories)]:
#                     rgb = colors.to_rgb(color)
#                     cvd_rgb = simulate_colorblindness(rgb, cvd)
#                     cvd_colors.append(cvd_rgb)
#                 axes[i].bar(categories, values, color=cvd_colors)
        
#         axes[i].set_title(f'{cvd.capitalize()}')
#         axes[i].set_ylim(0, 60)
#         axes[i].tick_params(axis='x', rotation=45)
    
#     plt.suptitle(f'Bar Chart Appearance with Different Color Vision Conditions', fontsize=14)
#     plt.tight_layout()
#     plt.show()

# # Compare standard palette vs. colorblind-friendly palette
# standard_palette = ['red', 'green', 'blue', 'yellow', 'purple']
# cb_palette = ['#0072B2', '#009E73', '#D55E00', '#CC79A7', '#F0E442']

# plt.figure(figsize=(10, 6))
# plt.subplot(2, 1, 1)
# plt.title('Standard Red-Green Palette (Problematic for Colorblind Viewers)', fontsize=12)
# plt.bar(categories, values, color=standard_palette)

# plt.subplot(2, 1, 2)
# plt.title('Colorblind-Friendly Palette', fontsize=12)
# plt.bar(categories, values, color=cb_palette)

# plt.tight_layout()
# plt.show()

# # Show how each palette appears with different types of color blindness
# print("Standard Palette Under Different Color Vision Conditions:")
# compare_cvd_perception(standard_palette)

# print("\nColorblind-Friendly Palette Under Different Color Vision Conditions:")
# compare_cvd_perception(cb_palette)

```
ValueError: No path found from {'name': 'sRGB1+D65:deuteranomaly'} -> {'name': 'sRGB1+D65'}
```

In [ ]:
# # ---------------------------- imports ----------------------------
# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib import colors

# # --------------------------- data -------------------------------
# categories = ['Category A','Category B','Category C','Category D','Category E']
# values     = [25, 40, 30, 55, 15]

# standard_palette = ['red','green','blue','yellow','purple']
# cb_palette       = ['#0072B2','#009E73','#D55E00','#CC79A7','#F0E442']

# # ------------ Brettel 1997 × Machado 2009 matrices --------------
# # sRGB (linear)‑to‑linear‑sRGB transform for 100 % anomaly
# MATRIX = {
#     'deuteranomaly': np.array([[0.367, 0.861, -0.228],
#                                [0.280, 0.673,  0.047],
#                                [-0.012, 0.043, 0.969]]),
#     'protanomaly'  : np.array([[0.152, 1.053, -0.205],
#                                [0.115, 0.786,  0.099],
#                                [-0.004, -0.048, 1.052]]),
#     'tritanomaly'  : np.array([[1.256, -0.078, -0.178],
#                                [-0.078, 0.931,  0.148],
#                                [0.004, 0.691,  0.305]])
# }

# # ------------- helper: simulate CVD via matrix ------------------
# def simulate_colorblindness(rgb, cvd_type):
#     """
#     rgb: iterable of 3 floats (0‑1, sRGB)
#     cvd_type: 'original' | 'deuteranomaly' | 'protanomaly' | 'tritanomaly'
#     """
#     if cvd_type == 'original':
#         return np.asarray(rgb)

#     # 1) convert gamma‑encoded sRGB→linear
#     rgb = np.asarray(rgb)
#     lin  = np.where(rgb <= 0.04045,
#                     rgb/12.92,
#                     ((rgb+0.055)/1.055)**2.4)

#     # 2) apply 3×3 deficiency matrix
#     lin_cvd = MATRIX[cvd_type] @ lin

#     # 3) back to gamma‑encoded sRGB
#     rgb_cvd = np.where(lin_cvd <= 0.0031308,
#                        12.92*lin_cvd,
#                        1.055*lin_cvd**(1/2.4) - 0.055)

#     # clip to [0,1] just in case
#     return np.clip(rgb_cvd, 0, 1)

# # ---------------- comparison plotting ---------------------------
# def compare_cvd_perception(palette):
#     cvd_types = ['original','deuteranomaly','protanomaly','tritanomaly']
#     fig, axes = plt.subplots(1, 4, figsize=(18, 5))

#     for ax, cvd in zip(axes, cvd_types):
#         disp_colors = [
#             simulate_colorblindness(colors.to_rgb(c), cvd)
#             for c in palette
#         ]
#         ax.bar(categories, values, color=disp_colors)
#         ax.set_title(cvd.capitalize())
#         ax.set_ylim(0, 60)
#         ax.tick_params(axis='x', rotation=45)

#     plt.suptitle('Appearance Under Different Color‑Vision Conditions',
#                  fontsize=14)
#     plt.tight_layout()
#     plt.show()

# # ------------- show palettes & simulations ----------------------
# plt.figure(figsize=(9, 6))
# plt.subplot(2,1,1); plt.title('Standard Palette')
# plt.bar(categories, values, color=standard_palette)
# plt.subplot(2,1,2); plt.title('Color‑Blind‑Friendly Palette')
# plt.bar(categories, values, color=cb_palette)
# plt.tight_layout(); plt.show()

# print("Standard palette under CVD simulation:")
# compare_cvd_perception(standard_palette)

# print("\nColor‑blind‑friendly palette under CVD simulation:")
# compare_cvd_perception(cb_palette)

```
UserWarning: Glyph 8209 (\N{NON-BREAKING HYPHEN}) missing from font(s) Arial.

RuntimeWarning: invalid value encountered in power
  1.055*lin_cvd**(1/2.4) - 0.055)
```

In [ ]:
# ---------------------------- imports ----------------------------
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors

# --------------------------- data -------------------------------
categories = ['Category A','Category B','Category C','Category D','Category E']
values     = [25, 40, 30, 55, 15]

standard_palette = ['red','green','blue','yellow','purple']
cb_palette       = ['#0072B2','#009E73','#D55E00','#CC79A7','#F0E442']

# ------------ Brettel 1997 × Machado 2009 matrices --------------
MATRIX = {
    'deuteranomaly': np.array([[0.367, 0.861, -0.228],
                               [0.280, 0.673,  0.047],
                               [-0.012, 0.043, 0.969]]),
    'protanomaly'  : np.array([[0.152, 1.053, -0.205],
                               [0.115, 0.786,  0.099],
                               [-0.004, -0.048, 1.052]]),
    'tritanomaly'  : np.array([[1.256, -0.078, -0.178],
                               [-0.078, 0.931,  0.148],
                               [0.004, 0.691,  0.305]])
}

# ------------- helper: simulate CVD via matrix ------------------
def simulate_colorblindness(rgb, cvd_type):
    if cvd_type == 'original':
        return np.asarray(rgb)

    rgb = np.asarray(rgb)
    lin = np.where(rgb <= 0.04045, rgb/12.92, ((rgb+0.055)/1.055)**2.4)
    lin_cvd = MATRIX[cvd_type] @ lin
    lin_cvd = np.maximum(lin_cvd, 0.0)          # avoid negatives
    rgb_cvd = np.where(lin_cvd <= 0.0031308,
                       12.92*lin_cvd,
                       1.055*lin_cvd**(1/2.4) - 0.055)
    return np.clip(rgb_cvd, 0, 1)

# ---------------- comparison plotting ---------------------------
def compare_cvd_perception(palette):
    cvd_types = ['original','deuteranomaly','protanomaly','tritanomaly']
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))

    for ax, cvd in zip(axes, cvd_types):
        disp_colors = [
            simulate_colorblindness(colors.to_rgb(c), cvd)
            for c in palette
        ]
        ax.bar(categories, values, color=disp_colors)
        ax.set_title(cvd.capitalize())
        ax.set_ylim(0, 60)
        ax.tick_params(axis='x', rotation=45)

    plt.suptitle('Appearance Under Different Color-Vision Conditions', fontsize=14)
    plt.tight_layout()
    plt.show()

# ------------- show palettes & simulations ----------------------
plt.figure(figsize=(9, 6))
plt.subplot(2,1,1); plt.title('Standard Palette')
plt.bar(categories, values, color=standard_palette)
plt.subplot(2,1,2); plt.title('Color-Blind-Friendly Palette')
plt.bar(categories, values, color=cb_palette)
plt.tight_layout(); plt.show()

print("Standard palette under CVD simulation:")
compare_cvd_perception(standard_palette)

print("\nColor-blind-friendly palette under CVD simulation:")
compare_cvd_perception(cb_palette)

### Additional Techniques to Improve Accessibility

In addition to choosing colorblind-friendly colors, we can use other visual cues to enhance accessibility:

In [ ]:
# Create sample data for multiple data series
x = np.arange(1, 6)
data1 = [5, 15, 10, 20, 25]
data2 = [10, 20, 15, 25, 30]
data3 = [15, 10, 25, 30, 20]

plt.figure(figsize=(15, 10))

# 1. Use patterns in addition to color
plt.subplot(2, 2, 1)
bars = plt.bar(x, data1, color='#0072B2')
plt.bar(x + 0.25, data2, color='#D55E00')
plt.bar(x + 0.5, data3, color='#009E73')

plt.title('Color Only (Potential Issue)')
plt.xticks(x + 0.25, ['A', 'B', 'C', 'D', 'E'])
plt.legend(['Series 1', 'Series 2', 'Series 3'])

# 2. Use patterns in addition to color
plt.subplot(2, 2, 2)

# Pattern-filled bars using hatching
bars1 = plt.bar(x, data1, color='#0072B2', hatch='/')
bars2 = plt.bar(x + 0.25, data2, color='#D55E00', hatch='\\')
bars3 = plt.bar(x + 0.5, data3, color='#009E73', hatch='x')

plt.title('Color + Patterns (More Accessible)')
plt.xticks(x + 0.25, ['A', 'B', 'C', 'D', 'E'])
plt.legend(['Series 1', 'Series 2', 'Series 3'])

# 3. Use direct labeling
plt.subplot(2, 2, 3)
ax = plt.gca()
bars1 = ax.bar(x, data1, color='#0072B2', width=0.25)
bars2 = ax.bar(x + 0.25, data2, color='#D55E00', width=0.25)
bars3 = ax.bar(x + 0.5, data3, color='#009E73', width=0.25)

# Add direct labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{int(height)}', ha='center', va='bottom')

plt.title('Direct Data Labeling (More Accessible)')
plt.xticks(x + 0.25, ['A', 'B', 'C', 'D', 'E'])
plt.legend(['Series 1', 'Series 2', 'Series 3'])

# 4. Use different marker shapes for line plots
plt.subplot(2, 2, 4)
plt.plot(x, data1, color='#0072B2', marker='o', markersize=10, label='Series 1')
plt.plot(x, data2, color='#D55E00', marker='s', markersize=10, label='Series 2')
plt.plot(x, data3, color='#009E73', marker='^', markersize=10, label='Series 3')

plt.title('Different Markers + Colors (More Accessible)')
plt.xticks(x, ['A', 'B', 'C', 'D', 'E'])
plt.legend()

plt.tight_layout()
plt.suptitle('Techniques to Improve Visualization Accessibility', fontsize=16, y=1.02)
plt.show()

## Using Color to Highlight Information

Color is a powerful tool for drawing attention to specific aspects of your data. Strategic use of color can help guide viewers to the most important information in your visualization.

### Techniques for Highlighting with Color

1. **Contrast with Background**: Use high contrast colors to make key data stand out
2. **Selective Color**: Use color for important data points and gray for contextual data
3. **Color Intensity**: Use varying color intensity to create a visual hierarchy
4. **Preattentive Processing**: Leverage colors that immediately draw attention

Let's explore these techniques:

In [ ]:
# Generate sample data
np.random.seed(42)
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
data = np.random.randint(50, 100, size=12)
data[7] = 110  # Create an outlier for August

plt.figure(figsize=(15, 12))

# 1. Selective color technique - highlight specific points
plt.subplot(2, 2, 1)
colors = ['#cccccc'] * 12
colors[7] = '#E31A1C'  # Highlight August in red
plt.bar(months, data, color=colors)
plt.title('Selective Color: Highlighting an Outlier', fontsize=12)
plt.ylabel('Value')

# 2. Color intensity for grouping
plt.subplot(2, 2, 2)
quarters = ['#4575B4'] * 3 + ['#91BFDB'] * 3 + ['#FC8D59'] * 3 + ['#D73027'] * 3
plt.bar(months, data, color=quarters)
plt.title('Color Intensity: Grouping by Quarter', fontsize=12)
plt.ylabel('Value')

# 3. Before-After Comparison
plt.subplot(2, 2, 3)
business_change = np.random.randint(60, 100, size=12)
business_change[6:] = business_change[6:] + 15  # Show improvement after policy change

# Color before and after different colors
colors_before_after = ['#AEB6BF'] * 6 + ['#5DADE2'] * 6
plt.bar(months, business_change, color=colors_before_after)
plt.axvline(x=5.5, color='black', linestyle='--', alpha=0.7)
plt.text(5.5, 105, 'Policy Change', rotation=90, va='bottom', ha='right')
plt.title('Highlighting Before/After Comparison', fontsize=12)
plt.ylabel('Business Metric')

# 4. Multi-variable highlighting
plt.subplot(2, 2, 4)
target = 80
performance = np.random.randint(60, 100, size=12)

# Create a diverging color scheme based on performance vs target
colors = []
for value in performance:
    if value >= target:
        # Above target (good) - green scale
        intensity = min(1, 0.4 + 0.6 * (value - target) / 30)
        colors.append((0, intensity, 0))
    else:
        # Below target (bad) - red scale
        intensity = min(1, 0.4 + 0.6 * (target - value) / 30)
        colors.append((intensity, 0, 0))

plt.bar(months, performance, color=colors)
plt.axhline(y=target, color='black', linestyle='--', alpha=0.7)
plt.text(0, target+1, f'Target: {target}', va='bottom', ha='left')
plt.title('Color Intensity to Show Performance vs Target', fontsize=12)
plt.ylabel('Performance')

plt.tight_layout()
plt.suptitle('Using Color to Highlight Information', fontsize=16, y=1.02)
plt.show()

### Advanced Highlighting Techniques

Let's explore some more advanced techniques for using color to highlight information:

In [ ]:
# Create more complex data for highlighting
np.random.seed(42)

# Create sample data for a time series with highlighted regions
dates = pd.date_range(start='2022-01-01', end='2022-12-31', freq='D')
ts_data = pd.Series(np.cumsum(np.random.normal(0, 1, len(dates))), index=dates)

# Create sample data for a heatmap with highlighted cells
corr_data = np.random.uniform(-0.5, 0.5, size=(10, 10))
np.fill_diagonal(corr_data, 1)
corr_data = (corr_data + corr_data.T) / 2  # Make it symmetric like a correlation matrix
corr_data[2, 5] = 0.9  # Add a strong correlation to highlight
corr_data[5, 2] = 0.9  # Add a strong correlation to highlight

plt.figure(figsize=(15, 12))

# 1. Highlight time periods with background color
plt.subplot(2, 2, 1)
plt.plot(ts_data.index, ts_data.values, color='#3498DB', linewidth=2)

# Add highlighted regions for important periods
highlight_periods = [
    ('2022-03-01', '2022-04-01', 'COVID-19 Surge', '#FADBD8'),
    ('2022-07-15', '2022-08-15', 'Summer Campaign', '#D5F5E3')
]

for start, end, label, color in highlight_periods:
    start_date = pd.Timestamp(start)
    end_date = pd.Timestamp(end)
    plt.axvspan(start_date, end_date, color=color, alpha=0.5, label=label)

plt.title('Highlighting Time Periods with Background Color', fontsize=12)
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()

# 2. Heatmap with custom colormap to highlight specific range
plt.subplot(2, 2, 2)
# Create a custom colormap that highlights strong correlations
colors = [(0.8, 0, 0), (1, 1, 1), (0, 0, 0.8)]  # red -> white -> blue
positions = [0, 0.5, 1]
cmap = LinearSegmentedColormap.from_list('highlight_cmap', list(zip(positions, colors)))

sns.heatmap(corr_data, cmap=cmap, vmin=-1, vmax=1, 
           annot=True, fmt='.2f', linewidths=0.5)
plt.title('Heatmap with Custom Colormap Highlighting Strong Correlations', fontsize=12)

# 3. Focus + Context visualization using color
plt.subplot(2, 2, 3)
# Create sample categorical data
categories = ['Cat A', 'Cat B', 'Cat C', 'Cat D', 'Cat E', 'Cat F', 'Cat G', 'Cat H']
values = np.random.randint(10, 100, size=8)

# Define focus category
focus_cat = 'Cat D'
focus_idx = categories.index(focus_cat)

# Create color list - focus gets bright color, context gets muted colors
colors = ['#D3D3D3'] * len(categories)  # Start with all light gray
colors[focus_idx] = '#E74C3C'  # Bright red for focus

plt.bar(categories, values, color=colors)
plt.title('Focus + Context: Highlighting a Specific Category', fontsize=12)
plt.ylabel('Value')
plt.xticks(rotation=45)

# 4. Color-coded data points based on threshold
plt.subplot(2, 2, 4)
# Sample scatter data
x = np.random.normal(size=100)
y = x * 0.7 + np.random.normal(size=100, scale=0.5)
threshold = 1.0

# Color points based on distance from origin
distances = np.sqrt(x**2 + y**2)
colors = np.where(distances > threshold, '#E74C3C', '#3498DB')  # Red if beyond threshold, blue otherwise

plt.scatter(x, y, c=colors, s=70, alpha=0.7)
circle = plt.Circle((0, 0), threshold, fill=False, color='black', linestyle='--')
plt.gca().add_patch(circle)
plt.title('Color-Coded Data Points Based on Threshold', fontsize=12)
plt.xlabel('X Value')
plt.ylabel('Y Value')
plt.axhline(y=0, color='gray', alpha=0.3)
plt.axvline(x=0, color='gray', alpha=0.3)
plt.xlim(-2.5, 2.5)
plt.ylim(-2.5, 2.5)

plt.tight_layout()
plt.suptitle('Advanced Techniques for Highlighting Information with Color', fontsize=16, y=1.02)
plt.show()

## Interactive Color Selection with Plotly

Plotly offers powerful capabilities for creating interactive visualizations with thoughtful color schemes. Let's explore how to work with color in Plotly for both static and interactive visualizations.

### Key Plotly Color Features

1. **Built-in color sequences**: `px.colors.sequential`, `px.colors.diverging`, `px.colors.cyclical`, `px.colors.qualitative`
2. **Color scales**: Continuous color mapping based on variable values
3. **Custom color mapping**: Assign specific colors to specific categories or values
4. **Interactive color legends**: Filter data by clicking on color legend items

In [ ]:
# # Explore Plotly's built-in color palettes
# def display_plotly_color_scales():
#     """Display the built-in color scales in Plotly"""
#     # Get lists of color scales
#     sequential_scales = px.colors.named_colorscales()
#     qualitative_scales = list(px.colors.qualitative.swatches().keys())
    
#     print(f"Plotly has {len(sequential_scales)} sequential/diverging color scales")
#     print(f"Plotly has {len(qualitative_scales)} qualitative color scales")
    
#     # Display some selected color scales
#     selected_sequential = ['Viridis', 'Plasma', 'Inferno', 'Turbo', 'Blues', 'Greens']
#     selected_diverging = ['RdBu', 'BrBG', 'PiYG', 'PRGn', 'PuOr', 'RdYlGn']
#     selected_qualitative = ['Plotly', 'D3', 'G10', 'T10', 'Alphabet', 'Dark24']
    
#     fig = make_subplots(rows=3, cols=1, 
#                       subplot_titles=("Sequential Color Scales", 
#                                      "Diverging Color Scales", 
#                                      "Qualitative Color Scales"),
#                       vertical_spacing=0.12)
    
#     # Create color scale swatches
#     for i, scale in enumerate(selected_sequential):
#         color_vals = px.colors.sample_colorscale(scale, [j/10 for j in range(11)])
#         for j, color in enumerate(color_vals):
#             fig.add_trace(
#                 go.Bar(
#                     x=[j], y=[1], 
#                     marker_color=color,
#                     name=scale if j == 0 else None,
#                     showlegend=True if j == 0 else False,
#                     hoverinfo='name'
#                 ),
#                 row=1, col=1
#             )
    
#     # Create diverging scale swatches
#     for i, scale in enumerate(selected_diverging):
#         color_vals = px.colors.sample_colorscale(scale, [j/10 for j in range(11)])
#         for j, color in enumerate(color_vals):
#             fig.add_trace(
#                 go.Bar(
#                     x=[j], y=[1], 
#                     marker_color=color,
#                     name=scale if j == 0 else None,
#                     showlegend=True if j == 0 else False,
#                     hoverinfo='name'
#                 ),
#                 row=2, col=1
#             )
    
#     # Create qualitative color swatches
#     for i, palette_name in enumerate(selected_qualitative):
#         palette = getattr(px.colors.qualitative, palette_name)
#         for j, color in enumerate(palette[:10]):  # Limit to 10 colors
#             fig.add_trace(
#                 go.Bar(
#                     x=[j], y=[1], 
#                     marker_color=color,
#                     name=palette_name if j == 0 else None,
#                     showlegend=True if j == 0 else False,
#                     hoverinfo='name'
#                 ),
#                 row=3, col=1
#             )
    
#     # Update layout
#     fig.update_layout(
#         height=700,
#         title_text="Plotly Color Scale Examples",
#         barmode='stack',
#         yaxis=dict(showticklabels=False, showgrid=False),
#         yaxis2=dict(showticklabels=False, showgrid=False),
#         yaxis3=dict(showticklabels=False, showgrid=False),
#         xaxis=dict(showticklabels=False, showgrid=False),
#         xaxis2=dict(showticklabels=False, showgrid=False),
#         xaxis3=dict(showticklabels=False, showgrid=False),
#         plot_bgcolor='rgba(0,0,0,0)'
#     )
    
#     fig.show()

# # Display Plotly color scales
# display_plotly_color_scales()

```
AttributeError: 'Figure' object has no attribute 'keys'
```

In [ ]:
# Explore Plotly's built-in color palettes
def display_plotly_color_scales():
    """Display the built-in color scales in Plotly"""
    # Get lists of color scales
    sequential_scales = px.colors.named_colorscales()
    # Get the qualitative color scales directly from px.colors.qualitative
    qualitative_scales = list(px.colors.qualitative.__dict__.keys())

    print(f"Plotly has {len(sequential_scales)} sequential/diverging color scales")
    print(f"Plotly has {len(qualitative_scales)} qualitative color scales")
    
    # Display some selected color scales
    selected_sequential = ['Viridis', 'Plasma', 'Inferno', 'Turbo', 'Blues', 'Greens']
    selected_diverging = ['RdBu', 'BrBG', 'PiYG', 'PRGn', 'PuOr', 'RdYlGn']
    selected_qualitative = ['Plotly', 'D3', 'G10', 'T10', 'Alphabet', 'Dark24']
    
    fig = make_subplots(rows=3, cols=1, 
                      subplot_titles=("Sequential Color Scales", 
                                     "Diverging Color Scales", 
                                     "Qualitative Color Scales"),
                      vertical_spacing=0.12)
    
    # Create color scale swatches
    for i, scale in enumerate(selected_sequential):
        color_vals = px.colors.sample_colorscale(scale, [j/10 for j in range(11)])
        for j, color in enumerate(color_vals):
            fig.add_trace(
                go.Bar(
                    x=[j], y=[1], 
                    marker_color=color,
                    name=scale if j == 0 else None,
                    showlegend=True if j == 0 else False,
                    hoverinfo='name'
                ),
                row=1, col=1
            )
    
    # Create diverging scale swatches
    for i, scale in enumerate(selected_diverging):
        color_vals = px.colors.sample_colorscale(scale, [j/10 for j in range(11)])
        for j, color in enumerate(color_vals):
            fig.add_trace(
                go.Bar(
                    x=[j], y=[1], 
                    marker_color=color,
                    name=scale if j == 0 else None,
                    showlegend=True if j == 0 else False,
                    hoverinfo='name'
                ),
                row=2, col=1
            )
    
    # Create qualitative color swatches
    for i, palette_name in enumerate(selected_qualitative):
        palette = getattr(px.colors.qualitative, palette_name)
        for j, color in enumerate(palette[:10]):  # Limit to 10 colors
            fig.add_trace(
                go.Bar(
                    x=[j], y=[1], 
                    marker_color=color,
                    name=palette_name if j == 0 else None,
                    showlegend=True if j == 0 else False,
                    hoverinfo='name'
                ),
                row=3, col=1
            )
    
    # Update layout
    fig.update_layout(
        height=700,
        title_text="Plotly Color Scale Examples",
        barmode='stack',
        yaxis=dict(showticklabels=False, showgrid=False),
        yaxis2=dict(showticklabels=False, showgrid=False),
        yaxis3=dict(showticklabels=False, showgrid=False),
        xaxis=dict(showticklabels=False, showgrid=False),
        xaxis2=dict(showticklabels=False, showgrid=False),
        xaxis3=dict(showticklabels=False, showgrid=False),
        plot_bgcolor='rgba(0,0,0,0)'
    )
    
    fig.show()

# Display Plotly color scales
display_plotly_color_scales()

### Creating Interactive Color-Based Visualizations in Plotly

Now let's apply what we've learned about Plotly's color capabilities to create interactive visualizations. Plotly's interactive features allow users to:

1. **Hover over data points** to get color-coded information
2. **Filter data by color groups** by clicking on legend items
3. **Explore relationships** between variables using color as a dimension
4. **Zoom into color regions** for detailed analysis

Let's build some examples that showcase these features:

In [ ]:
# Create sample data for interactive visualizations
np.random.seed(42)

# Create a dataset for scatter plot
n = 200
df_scatter = pd.DataFrame({
    'x': np.random.normal(0, 2, n),
    'y': np.random.normal(0, 2, n),
    'size': np.random.uniform(5, 30, n),
    'category': np.random.choice(['Group A', 'Group B', 'Group C', 'Group D'], n),
    'value': np.random.uniform(0, 100, n)
})

# Create a dataset for bar chart
categories = ['Technology', 'Healthcare', 'Finance', 'Consumer Goods', 'Energy', 'Materials']
years = [2018, 2019, 2020, 2021, 2022]
data_values = []
for year in years:
    values = np.random.randint(20, 100, len(categories))
    for cat, val in zip(categories, values):
        data_values.append({'Category': cat, 'Year': year, 'Revenue': val})
df_bar = pd.DataFrame(data_values)

# 1. Interactive scatter plot with color dimension
def create_interactive_scatter():
    # Create scatter plot with color based on category and size based on value
    fig = px.scatter(
        df_scatter, x='x', y='y', 
        color='category',
        size='size',
        hover_name='category',
        hover_data={'x': False, 'y': False, 'size': False, 'category': False, 'value': True},
        labels={'value': 'Score', 'category': 'Group'},
        color_discrete_sequence=px.colors.qualitative.G10,
        title='Interactive Color-Based Scatter Plot'
    )
    
    # Customize the hover template
    fig.update_traces(
        hovertemplate='<b>%{hovertext}</b><br>Score: %{customdata[0]:.1f}<extra></extra>'
    )
    
    # Update layout
    fig.update_layout(
        legend_title='Group',
        height=600,
        plot_bgcolor='rgba(240, 240, 240, 0.8)'
    )
    
    return fig

# 2. Interactive bar chart with color dimension
def create_interactive_bar():
    # Create bar chart with color based on category
    fig = px.bar(
        df_bar, 
        x='Category', 
        y='Revenue', 
        color='Year',
        barmode='group',
        color_discrete_sequence=px.colors.sequential.Plasma_r,
        title='Revenue by Category (Click on legend items to filter)',
        labels={'Revenue': 'Annual Revenue ($M)', 'Year': 'Year'}
    )
    
    # Customize hover information
    fig.update_traces(
        hovertemplate='<b>%{x}</b><br>Year: %{marker.color}<br>Revenue: $%{y}M<extra></extra>'
    )
    
    # Update layout
    fig.update_layout(
        legend_title='Year',
        height=500,
        plot_bgcolor='rgba(240, 240, 240, 0.8)'
    )
    
    return fig

# Display the interactive visualizations
scatter_fig = create_interactive_scatter()
scatter_fig.show()

bar_fig = create_interactive_bar()
bar_fig.show()

### Color-Based Interactions and Hover Information

One of Plotly's strengths is using color to enhance interactive data exploration. Let's create more advanced examples that leverage color for interactive insights:

1. **Color-based hover information** - Customize what appears in hover tooltips based on color values
2. **Color legends as interactive filters** - Use color legends for dynamic data filtering
3. **Color dimension with animation** - Add a time dimension to see how colors change over time

In [ ]:
# Create more advanced interactive color-based visualizations

# 1. Heatmap with custom hover information
def create_interactive_heatmap():
    # Create correlation matrix
    np.random.seed(42)
    feature_names = ['Feature A', 'Feature B', 'Feature C', 'Feature D', 'Feature E', 
                    'Feature F', 'Feature G', 'Feature H']
    
    # Generate a random correlation matrix
    corr_matrix = np.random.uniform(-0.7, 0.7, (len(feature_names), len(feature_names)))
    # Make it symmetric
    corr_matrix = (corr_matrix + corr_matrix.T) / 2
    # Set diagonal to 1
    np.fill_diagonal(corr_matrix, 1)
    
    # Create a dataframe
    corr_df = pd.DataFrame(corr_matrix, columns=feature_names, index=feature_names)
    
    # Create interactive heatmap
    fig = px.imshow(
        corr_df,
        color_continuous_scale='RdBu_r',
        zmin=-1, zmax=1,
        title='Interactive Correlation Heatmap with Color-Coded Hover',
        labels={'color': 'Correlation'}
    )
    
    # Customize hover template
    fig.update_traces(
        hovertemplate='%{y} & %{x}<br>Correlation: %{z:.3f}<extra></extra>',
        # Color-code text based on value (dark for light cells, light for dark cells)
        texttemplate='%{z:.2f}',
        textfont={"color":"white"}
    )
    
    # Update layout
    fig.update_layout(
        height=550,
        width=650
    )
    
    return fig

# 2. Bubble chart with multiple color dimensions and interactive legend
def create_interactive_bubble_chart():
    # Create country data with region-based coloring
    np.random.seed(42)
    regions = ['North America', 'Europe', 'Asia', 'Africa', 'South America', 'Oceania']
    countries = []
    
    for region in regions:
        n_countries = np.random.randint(5, 10)
        for i in range(n_countries):
            gdp = np.random.uniform(500, 50000)
            life_exp = 40 + 40 * (gdp/50000)**0.4 + np.random.normal(0, 3)
            population = np.random.randint(1, 1500)
            growth = np.random.uniform(-2, 8)
            
            countries.append({
                'Country': f'{region} Country {i+1}',
                'Region': region,
                'GDP_per_capita': gdp,
                'Life_expectancy': life_exp,
                'Population': population,
                'Growth_rate': growth
            })
    
    df_countries = pd.DataFrame(countries)
    
    # Create interactive bubble chart
    fig = px.scatter(
        df_countries, 
        x='GDP_per_capita', 
        y='Life_expectancy', 
        size='Population', 
        color='Region',
        hover_name='Country',
        hover_data={'Region': True, 'GDP_per_capita': ':.0f', 'Life_expectancy': ':.1f', 
                   'Population': True, 'Growth_rate': ':.1f'},
        color_discrete_sequence=px.colors.qualitative.Bold,
        size_max=50,
        title='Interactive Bubble Chart with Region-Based Coloring',
        labels={
            'GDP_per_capita': 'GDP per Capita ($)',
            'Life_expectancy': 'Life Expectancy (years)',
            'Population': 'Population (millions)',
            'Growth_rate': 'Growth Rate (%)'
        }
    )
    
    # Update layout
    fig.update_layout(
        height=600,
        legend_title='Region',
        plot_bgcolor='rgba(240, 240, 240, 0.8)'
    )
    
    # Update axes
    fig.update_xaxes(type='log', range=[2.5, 4.8])
    
    return fig

# 3. Animated scatter plot with changing colors over time
def create_animated_color_chart():
    # Create data for animation
    np.random.seed(42)
    n_points = 50
    n_frames = 20
    
    # Base position and category
    base_x = np.random.normal(0, 1, n_points)
    base_y = np.random.normal(0, 1, n_points)
    categories = np.random.choice(['Group 1', 'Group 2', 'Group 3'], n_points)
    
    # Create empty lists to store data
    frames_data = []
    
    # Create data for each frame
    for frame in range(n_frames):
        # Values drift over time with some randomness
        x = base_x + 0.1 * frame * np.random.normal(0, 0.1, n_points)
        y = base_y + 0.1 * frame * np.random.normal(0, 0.1, n_points)
        
        # Values increase over time with category-specific patterns
        values = []
        for cat in categories:
            if cat == 'Group 1':
                val = 20 + frame*3 + np.random.normal(0, 5)
            elif cat == 'Group 2':
                val = 15 + frame*2 + np.random.normal(0, 4)
            else:
                val = 10 + frame*1.5 + np.random.normal(0, 3)
            values.append(val)
        
        for i in range(n_points):
            frames_data.append({
                'x': x[i],
                'y': y[i],
                'category': categories[i],
                'value': values[i],
                'frame': frame
            })
    
    # Create DataFrame
    df_animation = pd.DataFrame(frames_data)
    
    # Create animated scatter plot
    fig = px.scatter(
        df_animation,
        x='x', 
        y='y', 
        color='value',
        color_continuous_scale='Viridis',
        animation_frame='frame',
        range_color=[0, 80],
        size='value',
        size_max=40,
        hover_name='category',
        title='Animated Color Changes Over Time',
        labels={'value': 'Value', 'category': 'Group', 'x': 'X Position', 'y': 'Y Position'},
    )
    
    # Update layout
    fig.update_layout(
        height=600,
        coloraxis_colorbar=dict(title='Value'),
        plot_bgcolor='rgba(240, 240, 240, 0.8)'
    )
    
    # Update animation settings
    fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 400
    
    return fig

# Display interactive visualizations
heatmap_fig = create_interactive_heatmap()
heatmap_fig.show()

bubble_fig = create_interactive_bubble_chart()
bubble_fig.show()

animated_fig = create_animated_color_chart()
animated_fig.show()

### Customizing Color Scales in Plotly

Plotly offers powerful tools for creating custom color scales and mappings. This is especially useful when:

1. **Corporate branding**: You need to align visualization colors with brand guidelines
2. **Special color needs**: Specific colors have domain significance in your field
3. **Enhanced readability**: Default scales need tweaking for clarity or accessibility
4. **Multi-dimensional data**: You need to encode multiple variables through color

Let's explore how to create custom color scales and mappings in Plotly:

In [ ]:
# # Creating and using custom color scales in Plotly

# # 1. Create custom sequential color scale
# def create_custom_color_scales():
#     """Demonstrate how to create custom color scales in Plotly"""
    
#     # Create sample data
#     np.random.seed(42)
#     z = np.random.randint(0, 100, (20, 20))
    
#     # Default color scale for comparison
#     fig_default = px.imshow(
#         z, 
#         title="Default 'Viridis' Color Scale",
#         color_continuous_scale='Viridis'
#     )
    
#     fig_default.update_layout(height=400)
#     fig_default.show()
    
#     # 1. Custom sequential scale from RGB values
#     custom_seq = [
#         [0, 'rgb(255, 255, 230)'],        # Very light yellow
#         [0.2, 'rgb(217, 240, 163)'],      # Light yellow-green
#         [0.4, 'rgb(173, 221, 142)'],      # Medium yellow-green
#         [0.6, 'rgb(120, 198, 121)'],      # Medium green
#         [0.8, 'rgb(49, 163, 84)'],        # Dark green
#         [1, 'rgb(0, 104, 55)']            # Very dark green
#     ]
    
#     fig_custom_seq = px.imshow(
#         z, 
#         title="Custom Sequential Scale",
#         color_continuous_scale=custom_seq
#     )
    
#     fig_custom_seq.update_layout(height=400)
#     fig_custom_seq.show()
    
#     # 2. Custom diverging scale from hex values
#     custom_div = [
#         [0, '#2166AC'],                  # Dark blue
#         [0.2, '#67A9CF'],                # Medium blue
#         [0.4, '#D1E5F0'],                # Light blue
#         [0.5, '#F7F7F7'],                # White
#         [0.6, '#FDDBC7'],                # Light red
#         [0.8, '#EF8A62'],                # Medium red
#         [1, '#B2182B']                   # Dark red
#     ]
    
#     # Create data with a midpoint
#     z_diverging = z - 50  # Center around 0
    
#     fig_custom_div = px.imshow(
#         z_diverging, 
#         title="Custom Diverging Scale",
#         color_continuous_scale=custom_div,
#         color_continuous_midpoint=0
#     )
    
#     fig_custom_div.update_layout(height=400)
#     fig_custom_div.show()
    
#     # 3. Custom branding scale
#     # Assume these are company brand colors
#     brand_primary = '#1E88E5'    # Blue
#     brand_secondary = '#7CB342'  # Green
#     brand_accent = '#FFC107'     # Yellow/Gold
    
#     # Create a brand-aligned color scale
#     brand_scale = [
#         [0, brand_primary],
#         [0.5, brand_secondary],
#         [1, brand_accent]
#     ]
    
#     fig_brand = px.imshow(
#         z, 
#         title="Brand-Aligned Color Scale",
#         color_continuous_scale=brand_scale
#     )
    
#     fig_brand.update_layout(height=400)
#     fig_brand.show()
    
#     return custom_seq, custom_div, brand_scale

# # Call function to create and display custom color scales
# custom_seq, custom_div, brand_scale = create_custom_color_scales()

# # 2. Apply custom color maps to different plots
# def apply_custom_color_scales():
#     """Apply the custom color scales to different plot types"""
    
#     # Generate sample data
#     np.random.seed(42)
    
#     # Create a dataframe for a surface plot
#     x = np.linspace(-3, 3, 50)
#     y = np.linspace(-3, 3, 50)
#     x_grid, y_grid = np.meshgrid(x, y)
#     z_grid = np.sin(x_grid) * np.cos(y_grid)
    
#     # 1. Surface plot with custom sequential scale
#     fig_surface = go.Figure(data=[go.Surface(
#         z=z_grid,
#         x=x_grid,
#         y=y_grid,
#         colorscale=custom_seq
#     )])
    
#     fig_surface.update_layout(
#         title='3D Surface with Custom Sequential Color Scale',
#         scene=dict(
#             xaxis_title='X',
#             yaxis_title='Y',
#             zaxis_title='Z',
#         ),
#         width=700,
#         height=700
#     )
    
#     fig_surface.show()
    
#     # 2. Contour plot with custom diverging scale
#     fig_contour = go.Figure(data=go.Contour(
#         z=z_grid,
#         contours=dict(
#             showlabels=True,
#             labelfont=dict(size=12, color='white')
#         ),
#         colorscale=custom_div,
#         colorbar=dict(title='Value', titleside='right')
#     ))
    
#     fig_contour.update_layout(
#         title='Contour Plot with Custom Diverging Color Scale',
#         width=700,
#         height=600
#     )
    
#     fig_contour.show()
    
#     # 3. Continuous vs. Discrete Color Mapping
#     # Create sample data for scatter plot
#     n = 100
#     df_scatter = pd.DataFrame({
#         'x': np.random.normal(0, 1, n),
#         'y': np.random.normal(0, 1, n),
#         'value': np.random.uniform(-1, 1, n)
#     })
    
#     # Continuous color mapping
#     fig_continuous = px.scatter(
#         df_scatter, x='x', y='y', color='value',
#         color_continuous_scale=custom_div,
#         color_continuous_midpoint=0,
#         title='Continuous Color Mapping',
#         labels={'value': 'Value', 'x': 'X Position', 'y': 'Y Position'},
#         width=600, height=500
#     )
    
#     # Discrete color mapping (binned)
#     fig_discrete = px.scatter(
#         df_scatter, x='x', y='y', color='value',
#         color_continuous_scale=custom_div,
#         color_continuous_midpoint=0,
#         title='Discrete Color Mapping (Binned)',
#         labels={'value': 'Value', 'x': 'X Position', 'y': 'Y Position'},
#         width=600, height=500
#     )
    
#     # Make the color mapping discrete by setting coloraxis
#     fig_discrete.update_layout(
#         coloraxis=dict(
#             colorscale=custom_div,
#             colorbar=dict(title='Value', tickvals=[-1, -0.5, 0, 0.5, 1]),
#             cmin=-1,
#             cmax=1,
#             colorbar_tickmode='array',
#             cmid=0,
#             # The key part: setting the discrete color steps
#             showscale=True,
#             discretized=True
#         )
#     )
    
#     fig_continuous.show()
#     fig_discrete.show()

# # Apply the custom color scales to different plots
# apply_custom_color_scales()

# # 3. Multi-dimensional color visualization
# def multi_dimensional_color():
#     """Create visualizations that use color to represent multiple dimensions"""
    
#     # Create sample data with multiple dimensions
#     np.random.seed(42)
#     n = 300
    
#     # Create a dataframe with multiple metrics
#     df = pd.DataFrame({
#         'x': np.random.normal(0, 1, n),
#         'y': np.random.normal(0, 1, n),
#         'size': np.random.uniform(5, 30, n),
#         'category': np.random.choice(['A', 'B', 'C', 'D'], n),
#         'metric1': np.random.uniform(0, 100, n),
#         'metric2': np.random.uniform(-50, 50, n),
#         'metric3': np.random.exponential(10, n)
#     })
    
#     # 1. Using both marker color and marker line color to show two dimensions
#     fig_dual = px.scatter(
#         df, x='x', y='y', 
#         color='metric1',  # Fill color represents metric1
#         color_continuous_scale='Viridis',
#         size='size',
#         hover_name='category',
#         title='Dual-Dimension Color Encoding'
#     )
    
#     # Add line color to represent the second metric
#     fig_dual.update_traces(
#         marker=dict(
#             line=dict(
#                 width=2,
#                 color=df['metric2'],  # Line color represents metric2
#                 colorscale='RdBu',
#                 cauto=True,
#                 showscale=True,
#                 colorbar=dict(
#                     title='Metric 2',
#                     x=1.1  # Position the second colorbar
#                 )
#             )
#         ),
#         selector=dict(mode='markers')
#     )
    
#     # Update layout
#     fig_dual.update_layout(
#         coloraxis_colorbar_title='Metric 1',
#         height=600,
#         width=800
#     )
    
#     fig_dual.show()
    
#     # 2. Using a colormap where luminance (brightness) and hue encode different variables
#     # Create a custom color function that encodes two variables
#     def bivariate_color(val1, val2):
#         """Create colors where hue is determined by val1 and lightness by val2"""
#         import colorsys
        
#         # Normalize values to 0-1
#         norm_val1 = (val1 - df['metric1'].min()) / (df['metric1'].max() - df['metric1'].min())
#         norm_val2 = (val2 - df['metric2'].min()) / (df['metric2'].max() - df['metric2'].min())
        
#         # Generate colors: hue from val1, saturation constant, lightness from val2
#         colors = []
#         for h, l in zip(norm_val1, norm_val2):
#             # Convert HSL to RGB (hue in 0-1 range)
#             r, g, b = colorsys.hls_to_rgb(h, 0.2 + 0.6*l, 0.9)  
#             # Scale to 0-255 and convert to hex
#             hex_color = f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}'
#             colors.append(hex_color)
        
#         return colors
    
#     # Create bivariate colors
#     bivariate_colors = bivariate_color(df['metric1'], df['metric2'])
    
#     # Create scatter plot with bivariate colors
#     fig_bivariate = go.Figure()
    
#     fig_bivariate.add_trace(go.Scatter(
#         x=df['x'],
#         y=df['y'],
#         mode='markers',
#         marker=dict(
#             size=10,
#             color=bivariate_colors,
#         ),
#         text=[f'Category: {c}<br>Metric1: {m1:.1f}<br>Metric2: {m2:.1f}' 
#               for c, m1, m2 in zip(df['category'], df['metric1'], df['metric2'])],
#         hoverinfo='text'
#     ))
    
#     # Add a color legend
#     fig_bivariate.update_layout(
#         title='Bivariate Color Encoding (Hue = Metric1, Brightness = Metric2)',
#         height=600,
#         width=700,
#         showlegend=False,
#         annotations=[
#             dict(
#                 x=1.15,
#                 y=1.05,
#                 xref="paper",
#                 yref="paper",
#                 text="Metric1 = Hue<br>Metric2 = Brightness",
#                 showarrow=False,
#                 font=dict(size=12)
#             )
#         ]
#     )
    
#     fig_bivariate.show()

# # Create multi-dimensional color visualizations
# multi_dimensional_color()

```
ValueError: Invalid property specified for object of type plotly.graph_objs.contour.ColorBar: 'titleside'
Did you mean "title"?
```

In [ ]:
# # Correcting the code that applies the discrete color mapping to the scatter plot

# def apply_custom_color_scales():
#     """Apply the custom color scales to different plot types"""
    
#     # Generate sample data
#     np.random.seed(42)
    
#     # Create a dataframe for a surface plot
#     x = np.linspace(-3, 3, 50)
#     y = np.linspace(-3, 3, 50)
#     x_grid, y_grid = np.meshgrid(x, y)
#     z_grid = np.sin(x_grid) * np.cos(y_grid)
    
#     # 1. Surface plot with custom sequential scale
#     fig_surface = go.Figure(data=[go.Surface(
#         z=z_grid,
#         x=x_grid,
#         y=y_grid,
#         colorscale=custom_seq
#     )])
    
#     fig_surface.update_layout(
#         title='3D Surface with Custom Sequential Color Scale',
#         scene=dict(
#             xaxis_title='X',
#             yaxis_title='Y',
#             zaxis_title='Z',
#         ),
#         width=700,
#         height=700
#     )
    
#     fig_surface.show()
    
#     # 2. Contour plot with custom diverging scale
#     fig_contour = go.Figure(data=go.Contour(
#         z=z_grid,
#         contours=dict(
#             showlabels=True,
#             labelfont=dict(size=12, color='white')
#         ),
#         colorscale=custom_div,
#         colorbar=dict(title='Value')  # Removed titleside and used only 'title'
#     ))
    
#     fig_contour.update_layout(
#         title='Contour Plot with Custom Diverging Color Scale',
#         width=700,
#         height=600
#     )
    
#     fig_contour.show()
    
#     # 3. Continuous vs. Discrete Color Mapping
#     # Create sample data for scatter plot
#     n = 100
#     df_scatter = pd.DataFrame({
#         'x': np.random.normal(0, 1, n),
#         'y': np.random.normal(0, 1, n),
#         'value': np.random.uniform(-1, 1, n)
#     })
    
#     # Continuous color mapping
#     fig_continuous = px.scatter(
#         df_scatter, x='x', y='y', color='value',
#         color_continuous_scale=custom_div,
#         color_continuous_midpoint=0,
#         title='Continuous Color Mapping',
#         labels={'value': 'Value', 'x': 'X Position', 'y': 'Y Position'},
#         width=600, height=500
#     )
    
#     # Discrete color mapping (binned)
#     fig_discrete = px.scatter(
#         df_scatter, x='x', y='y', color='value',
#         color_continuous_scale=custom_div,
#         color_continuous_midpoint=0,
#         title='Discrete Color Mapping (Binned)',
#         labels={'value': 'Value', 'x': 'X Position', 'y': 'Y Position'},
#         width=600, height=500
#     )
    
#     # Make the color mapping discrete by setting coloraxis
#     fig_discrete.update_layout(
#         coloraxis=dict(
#             colorscale=custom_div,
#             colorbar=dict(title='Value', tickvals=[-1, -0.5, 0, 0.5, 1]),
#             cmin=-1,
#             cmax=1,
#             colorbar_tickmode='array',
#             cmid=0,  # Keep the midpoint for the diverging scale
#             showscale=True
#         )
#     )
    
#     fig_continuous.show()
#     fig_discrete.show()

# # Apply the custom color scales to different plots
# apply_custom_color_scales()

```
NameError: name 'custom_seq' is not defined
```

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# Correcting the code that applies the discrete color mapping to the scatter plot

def apply_custom_color_scales():
    """Apply the custom color scales to different plot types"""
    
    # Define custom color scales
    custom_seq = [
        [0, "rgb(255, 255, 255)"],  # White
        [0.5, "rgb(255, 127, 0)"],  # Orange
        [1, "rgb(255, 0, 0)"]       # Red
    ]
    
    custom_div = [
        [0, "rgb(0, 0, 255)"],   # Blue
        [0.5, "rgb(255, 255, 255)"],  # White
        [1, "rgb(255, 0, 0)"]    # Red
    ]
    
    # Generate sample data
    np.random.seed(42)
    
    # Create a dataframe for a surface plot
    x = np.linspace(-3, 3, 50)
    y = np.linspace(-3, 3, 50)
    x_grid, y_grid = np.meshgrid(x, y)
    z_grid = np.sin(x_grid) * np.cos(y_grid)
    
    # 1. Surface plot with custom sequential scale
    fig_surface = go.Figure(data=[go.Surface(
        z=z_grid,
        x=x_grid,
        y=y_grid,
        colorscale=custom_seq
    )])
    
    fig_surface.update_layout(
        title='3D Surface with Custom Sequential Color Scale',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
        ),
        width=700,
        height=700
    )
    
    fig_surface.show()
    
    # 2. Contour plot with custom diverging scale
    fig_contour = go.Figure(data=go.Contour(
        z=z_grid,
        contours=dict(
            showlabels=True,
            labelfont=dict(size=12, color='white')
        ),
        colorscale=custom_div,
        colorbar=dict(title='Value')  # Removed titleside and used only 'title'
    ))
    
    fig_contour.update_layout(
        title='Contour Plot with Custom Diverging Color Scale',
        width=700,
        height=600
    )
    
    fig_contour.show()
    
    # 3. Continuous vs. Discrete Color Mapping
    # Create sample data for scatter plot
    n = 100
    df_scatter = pd.DataFrame({
        'x': np.random.normal(0, 1, n),
        'y': np.random.normal(0, 1, n),
        'value': np.random.uniform(-1, 1, n)
    })
    
    # Continuous color mapping
    fig_continuous = px.scatter(
        df_scatter, x='x', y='y', color='value',
        color_continuous_scale=custom_div,
        color_continuous_midpoint=0,
        title='Continuous Color Mapping',
        labels={'value': 'Value', 'x': 'X Position', 'y': 'Y Position'},
        width=600, height=500
    )
    
    # Discrete color mapping (binned)
    fig_discrete = px.scatter(
        df_scatter, x='x', y='y', color='value',
        color_continuous_scale=custom_div,
        color_continuous_midpoint=0,
        title='Discrete Color Mapping (Binned)',
        labels={'value': 'Value', 'x': 'X Position', 'y': 'Y Position'},
        width=600, height=500
    )
    
    # Make the color mapping discrete by setting coloraxis
    fig_discrete.update_layout(
        coloraxis=dict(
            colorscale=custom_div,
            colorbar=dict(title='Value', tickvals=[-1, -0.5, 0, 0.5, 1]),
            cmin=-1,
            cmax=1,
            colorbar_tickmode='array',
            cmid=0,  # Keep the midpoint for the diverging scale
            showscale=True
        )
    )
    
    fig_continuous.show()
    fig_discrete.show()

# Apply the custom color scales to different plots
apply_custom_color_scales()

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import pandas as pd
import colorsys

# Creating and using custom color scales in Plotly

def create_custom_color_scales():
    """Demonstrate how to create custom color scales in Plotly"""
    
    # Create sample data
    np.random.seed(42)
    z = np.random.randint(0, 100, (20, 20))
    
    # Default color scale for comparison
    fig_default = px.imshow(
        z, 
        title="Default 'Viridis' Color Scale",
        color_continuous_scale='Viridis'
    )
    
    fig_default.update_layout(height=400)
    fig_default.show()
    
    # 1. Custom sequential scale from RGB values
    custom_seq = [
        [0, 'rgb(255, 255, 230)'],        # Very light yellow
        [0.2, 'rgb(217, 240, 163)'],      # Light yellow-green
        [0.4, 'rgb(173, 221, 142)'],      # Medium yellow-green
        [0.6, 'rgb(120, 198, 121)'],      # Medium green
        [0.8, 'rgb(49, 163, 84)'],        # Dark green
        [1, 'rgb(0, 104, 55)']            # Very dark green
    ]
    
    fig_custom_seq = px.imshow(
        z, 
        title="Custom Sequential Scale",
        color_continuous_scale=custom_seq
    )
    
    fig_custom_seq.update_layout(height=400)
    fig_custom_seq.show()
    
    # 2. Custom diverging scale from hex values
    custom_div = [
        [0, '#2166AC'],                  # Dark blue
        [0.2, '#67A9CF'],                # Medium blue
        [0.4, '#D1E5F0'],                # Light blue
        [0.5, '#F7F7F7'],                # White
        [0.6, '#FDDBC7'],                # Light red
        [0.8, '#EF8A62'],                # Medium red
        [1, '#B2182B']                   # Dark red
    ]
    
    # Create data with a midpoint
    z_diverging = z - 50  # Center around 0
    
    fig_custom_div = px.imshow(
        z_diverging, 
        title="Custom Diverging Scale",
        color_continuous_scale=custom_div,
        color_continuous_midpoint=0
    )
    
    fig_custom_div.update_layout(height=400)
    fig_custom_div.show()
    
    # 3. Custom branding scale
    # Assume these are company brand colors
    brand_primary = '#1E88E5'    # Blue
    brand_secondary = '#7CB342'  # Green
    brand_accent = '#FFC107'     # Yellow/Gold
    
    # Create a brand-aligned color scale
    brand_scale = [
        [0, brand_primary],
        [0.5, brand_secondary],
        [1, brand_accent]
    ]
    
    fig_brand = px.imshow(
        z, 
        title="Brand-Aligned Color Scale",
        color_continuous_scale=brand_scale
    )
    
    fig_brand.update_layout(height=400)
    fig_brand.show()
    
    return custom_seq, custom_div, brand_scale

# Call function to create and display custom color scales
custom_seq, custom_div, brand_scale = create_custom_color_scales()

# 2. Apply custom color maps to different plots
def apply_custom_color_scales():
    """Apply the custom color scales to different plot types"""
    
    # Generate sample data
    np.random.seed(42)
    
    # Create a dataframe for a surface plot
    x = np.linspace(-3, 3, 50)
    y = np.linspace(-3, 3, 50)
    x_grid, y_grid = np.meshgrid(x, y)
    z_grid = np.sin(x_grid) * np.cos(y_grid)
    
    # 1. Surface plot with custom sequential scale
    fig_surface = go.Figure(data=[go.Surface(
        z=z_grid,
        x=x_grid,
        y=y_grid,
        colorscale=custom_seq
    )])
    
    fig_surface.update_layout(
        title='3D Surface with Custom Sequential Color Scale',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
        ),
        width=700,
        height=700
    )
    
    fig_surface.show()
    
    # 2. Contour plot with custom diverging scale
    fig_contour = go.Figure(data=go.Contour(
        z=z_grid,
        contours=dict(
            showlabels=True,
            labelfont=dict(size=12, color='white')
        ),
        colorscale=custom_div,
        colorbar=dict(title='Value')  # Removed titleside and used only 'title'
    ))
    
    fig_contour.update_layout(
        title='Contour Plot with Custom Diverging Color Scale',
        width=700,
        height=600
    )
    
    fig_contour.show()
    
    # 3. Continuous vs. Discrete Color Mapping
    # Create sample data for scatter plot
    n = 100
    df_scatter = pd.DataFrame({
        'x': np.random.normal(0, 1, n),
        'y': np.random.normal(0, 1, n),
        'value': np.random.uniform(-1, 1, n)
    })
    
    # Continuous color mapping
    fig_continuous = px.scatter(
        df_scatter, x='x', y='y', color='value',
        color_continuous_scale=custom_div,
        color_continuous_midpoint=0,
        title='Continuous Color Mapping',
        labels={'value': 'Value', 'x': 'X Position', 'y': 'Y Position'},
        width=600, height=500
    )
    
    # Discrete color mapping (binned)
    fig_discrete = px.scatter(
        df_scatter, x='x', y='y', color='value',
        color_continuous_scale=custom_div,
        color_continuous_midpoint=0,
        title='Discrete Color Mapping (Binned)',
        labels={'value': 'Value', 'x': 'X Position', 'y': 'Y Position'},
        width=600, height=500
    )
    
    # Make the color mapping discrete by setting coloraxis
    fig_discrete.update_layout(
        coloraxis=dict(
            colorscale=custom_div,
            colorbar=dict(title='Value', tickvals=[-1, -0.5, 0, 0.5, 1]),
            cmin=-1,
            cmax=1,
            colorbar_tickmode='array',
            cmid=0,
            showscale=True
        )
    )
    
    fig_continuous.show()
    fig_discrete.show()

# Apply the custom color scales to different plots
apply_custom_color_scales()

# 3. Multi-dimensional color visualization
def multi_dimensional_color():
    """Create visualizations that use color to represent multiple dimensions"""
    
    # Create sample data with multiple dimensions
    np.random.seed(42)
    n = 300
    
    # Create a dataframe with multiple metrics
    df = pd.DataFrame({
        'x': np.random.normal(0, 1, n),
        'y': np.random.normal(0, 1, n),
        'size': np.random.uniform(5, 30, n),
        'category': np.random.choice(['A', 'B', 'C', 'D'], n),
        'metric1': np.random.uniform(0, 100, n),
        'metric2': np.random.uniform(-50, 50, n),
        'metric3': np.random.exponential(10, n)
    })
    
    # 1. Using both marker color and marker line color to show two dimensions
    fig_dual = px.scatter(
        df, x='x', y='y', 
        color='metric1',  # Fill color represents metric1
        color_continuous_scale='Viridis',
        size='size',
        hover_name='category',
        title='Dual-Dimension Color Encoding'
    )
    
    # Add line color to represent the second metric
    fig_dual.update_traces(
        marker=dict(
            line=dict(
                width=2,
                color=df['metric2'],  # Line color represents metric2
                colorscale='RdBu',
                cauto=True,
            )
        ),
        selector=dict(mode='markers')
    )
    
    # Update layout
    fig_dual.update_layout(
        coloraxis_colorbar_title='Metric 1',
        height=600,
        width=800
    )
    
    fig_dual.show()
    
    # 2. Using a colormap where luminance (brightness) and hue encode different variables
    # Create a custom color function that encodes two variables
    def bivariate_color(val1, val2):
        """Create colors where hue is determined by val1 and lightness by val2"""
        # Normalize values to 0-1
        norm_val1 = (val1 - df['metric1'].min()) / (df['metric1'].max() - df['metric1'].min())
        norm_val2 = (val2 - df['metric2'].min()) / (df['metric2'].max() - df['metric2'].min())
        
        # Generate colors: hue from val1, saturation constant, lightness from val2
        colors = []
        for h, l in zip(norm_val1, norm_val2):
            # Convert HSL to RGB (hue in 0-1 range)
            r, g, b = colorsys.hls_to_rgb(h, 0.2 + 0.6*l, 0.9)  
            # Scale to 0-255 and convert to hex
            hex_color = f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}'
            colors.append(hex_color)
        
        return colors
    
    # Create bivariate colors
    bivariate_colors = bivariate_color(df['metric1'], df['metric2'])
    
    # Create scatter plot with bivariate colors
    fig_bivariate = go.Figure()
    
    fig_bivariate.add_trace(go.Scatter(
        x=df['x'],
        y=df['y'],
        mode='markers',
        marker=dict(
            size=10,
            color=bivariate_colors,
        ),
        text=[f'Category: {c}<br>Metric1: {m1:.1f}<br>Metric2: {m2:.1f}' 
              for c, m1, m2 in zip(df['category'], df['metric1'], df['metric2'])],
        hoverinfo='text'
    ))
    
    # Add a color legend
    fig_bivariate.update_layout(
        title='Bivariate Color Encoding (Hue = Metric1, Brightness = Metric2)',
        height=600,
        width=700,
        showlegend=False,
        annotations=[
            dict(
                x=1.15,
                y=1.05,
                xref="paper",
                yref="paper",
                text="Metric1 = Hue<br>Metric2 = Brightness",
                showarrow=False,
                font=dict(size=12)
            )
        ]
    )
    
    fig_bivariate.show()

# Create multi-dimensional color visualizations
multi_dimensional_color()

## Best Practices for Color Usage

Effective color usage in data visualization requires careful consideration and planning. Here are key principles and best practices to follow:

### Core Principles for Effective Color Usage

1. **Purpose before aesthetics**: Choose colors that serve the data's purpose rather than just looking attractive
2. **Consistency**: Use consistent color schemes across related visualizations
3. **Restraint**: Limit the number of colors to avoid overwhelming the viewer
4. **Visual hierarchy**: Use color to guide attention to the most important information
5. **Accessibility**: Ensure visualizations work for people with color vision deficiencies

### Let's visualize these principles with examples:

In [ ]:
# # Best practices for color usage in visualizations

# # Create demonstration visualizations of color best practices
# def color_best_practices():
#     """Demonstrate best practices for color usage with examples"""
    
#     # Set random seed for reproducibility
#     np.random.seed(42)
    
#     # 1. CONSISTENCY: Good vs. Bad Examples
    
#     # Generate sample data for multiple charts
#     categories = ['A', 'B', 'C', 'D', 'E']
#     values1 = [23, 45, 56, 78, 42]
#     values2 = [35, 67, 21, 49, 63]
    
#     # Create subplots
#     fig = make_subplots(rows=2, cols=2, 
#                       subplot_titles=("Consistent Colors (Good)", "Inconsistent Colors (Bad)",
#                                      "Restrained Color Usage (Good)", "Too Many Colors (Bad)"),
#                       vertical_spacing=0.12,
#                       horizontal_spacing=0.1)
    
#     # Consistent color example (Good)
#     consistent_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    
#     fig.add_trace(
#         go.Bar(x=categories, y=values1, name='Dataset 1', marker_color=consistent_colors),
#         row=1, col=1
#     )
    
#     fig.add_trace(
#         go.Scatter(x=categories, y=values2, name='Dataset 2', mode='lines+markers',
#                   marker=dict(color=consistent_colors)),
#         row=1, col=1
#     )
    
#     # Inconsistent color example (Bad)
#     fig.add_trace(
#         go.Bar(x=categories, y=values1, name='Dataset 1', 
#               marker_color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']),
#         row=1, col=2
#     )
    
#     fig.add_trace(
#         go.Scatter(x=categories, y=values2, name='Dataset 2', mode='lines+markers',
#                   marker=dict(color=['#e377c2', '#8c564b', '#bcbd22', '#17becf', '#7f7f7f'])),
#         row=1, col=2
#     )
    
#     # 2. RESTRAINT: Good vs. Bad Examples
    
#     # Generate data for restrained example
#     timeseries_data = np.cumsum(np.random.normal(0, 1, 50))
#     threshold = 5
    
#     # Restrained example (Good) - Limited colors with clear purpose
#     x_data = list(range(len(timeseries_data)))
    
#     fig.add_trace(
#         go.Scatter(x=x_data, y=timeseries_data, name='Series', line=dict(color='#1f77b4')),
#         row=2, col=1
#     )
    
#     fig.add_trace(
#         go.Scatter(x=x_data, y=[threshold] * len(x_data), name='Threshold',
#                   line=dict(color='#d62728', dash='dash')),
#         row=2, col=1
#     )
    
#     # Areas above threshold - highlighting with color purpose
#     above_threshold = [max(y-threshold, 0) for y in timeseries_data]
#     x_above = [x for x, y in zip(x_data, timeseries_data) if y > threshold]
#     y_above = [y for y in timeseries_data if y > threshold]
    
#     fig.add_trace(
#         go.Scatter(x=x_above, y=y_above, name='Above Threshold',
#                   mode='markers', marker=dict(color='#d62728')),
#         row=2, col=1
#     )
    
#     # Too many colors example (Bad) - Using a different color for each data point
#     rainbow_colors = px.colors.sample_colorscale(
#         px.colors.sequential.Turbo, [i/49 for i in range(50)]
#     )
    
#     fig.add_trace(
#         go.Scatter(x=x_data, y=timeseries_data, name='Series',
#                   mode='markers+lines', 
#                   marker=dict(color=rainbow_colors, size=8),
#                   line=dict(color='gray')),
#         row=2, col=2
#     )
    
#     # Update layout
#     fig.update_layout(
#         height=800,
#         width=900,
#         title_text='Best Practices for Color Usage in Data Visualization',
#         showlegend=False
#     )
    
#     fig.show()
    
#     # 3. PURPOSE: Using color with clear purpose
    
#     # Create a dataset for sales performance
#     departments = ['Electronics', 'Clothing', 'Home Goods', 'Grocery', 'Beauty', 'Sports']
#     target_values = [100, 85, 70, 110, 90, 65]
#     actual_values = [95, 82, 60, 115, 95, 55]
    
#     performance = []
#     colors = []
#     for target, actual in zip(target_values, actual_values):
#         perf = (actual / target) * 100
#         performance.append(perf)
        
#         # Color based on performance
#         if perf >= 100:  # Met or exceeded target
#             colors.append('#2ecc71')  # Green
#         elif perf >= 90:  # Close to target
#             colors.append('#f39c12')  # Yellow/Orange
#         else:  # Below target
#             colors.append('#e74c3c')  # Red
    
#     # Create comparison figure
#     fig_purpose = make_subplots(rows=1, cols=2,
#                              subplot_titles=("Purposeful Color (Performance Indicator)", 
#                                           "Non-Purposeful Color (Decorative)"),
#                              horizontal_spacing=0.15)
    
#     # Purposeful color example
#     fig_purpose.add_trace(
#         go.Bar(
#             x=departments,
#             y=performance,
#             marker_color=colors,
#             text=[f"{p:.1f}%" for p in performance],
#             textposition='outside',
#             name='Performance'
#         ),
#         row=1, col=1
#     )
    
#     # Add a target line
#     fig_purpose.add_trace(
#         go.Scatter(
#             x=departments,
#             y=[100] * len(departments),
#             line=dict(color='black', dash='dash', width=1),
#             name='Target'
#         ),
#         row=1, col=1
#     )
    
#     # Non-purposeful color (decorative rainbow)
#     fig_purpose.add_trace(
#         go.Bar(
#             x=departments,
#             y=performance,
#             marker_color=px.colors.qualitative.Plotly,  # Random decorative colors
#             text=[f"{p:.1f}%" for p in performance],
#             textposition='outside',
#             name='Performance'
#         ),
#         row=1, col=2
#     )
    
#     # Add a target line
#     fig_purpose.add_trace(
#         go.Scatter(
#             x=departments,
#             y=[100] * len(departments),
#             line=dict(color='black', dash='dash', width=1),
#             name='Target'
#         ),
#         row=1, col=2
#     )
    
#     # Update layout
#     fig_purpose.update_layout(
#         height=400,
#         width=900,
#         title_text='Using Color with Clear Purpose',
#         yaxis_title='% of Target',
#         yaxis2_title='% of Target'
#     )
    
#     fig_purpose.update_yaxes(range=[50, 120], row=1, col=1)
#     fig_purpose.update_yaxes(range=[50, 120], row=1, col=2)
    
#     fig_purpose.show()

# # Generate best practices examples
# color_best_practices()

# # Creating a guide for choosing colors based on data types
# def color_selection_guide():
#     """Create a guide for selecting colors based on data types"""
    
#     # Generate sample data
#     np.random.seed(42)
    
#     # Create a figure with subplots for different data types
#     fig = make_subplots(
#         rows=3, cols=2,
#         subplot_titles=(
#             "Categorical Data (Qualitative)", 
#             "Sequential Data (Low to High)",
#             "Binary Data (Yes/No)", 
#             "Diverging Data (Negative to Positive)",
#             "Part-to-Whole Data (Proportions)",
#             "Highlight and Context Data"
#         ),
#         vertical_spacing=0.1,
#         horizontal_spacing=0.1
#     )
    
#     # 1. Categorical Data (Qualitative)
#     categories = ['Category A', 'Category B', 'Category C', 'Category D', 'Category E']
#     values = [45, 32, 38, 27, 42]
    
#     fig.add_trace(
#         go.Bar(
#             x=categories,
#             y=values,
#             marker_color=px.colors.qualitative.Safe,
#             name='Categories'
#         ),
#         row=1, col=1
#     )
    
#     # 2. Sequential Data (Low to High)
#     x = np.linspace(0, 10, 20)
#     y = np.linspace(0, 10, 20)
#     X, Y = np.meshgrid(x, y)
#     Z = np.sqrt(X**2 + Y**2)
    
#     fig.add_trace(
#         go.Heatmap(
#             z=Z,
#             colorscale='Viridis',
#             showscale=False
#         ),
#         row=1, col=2
#     )
    
#     # 3. Binary Data (Yes/No)
#     binary_categories = ['Group 1', 'Group 2', 'Group 3', 'Group 4', 'Group 5']
#     binary_values = [1, 0, 1, 0, 1]  # 1 = Yes, 0 = No
#     binary_colors = ['#2ecc71' if v == 1 else '#e74c3c' for v in binary_values]
    
#     fig.add_trace(
#         go.Bar(
#             x=binary_categories,
#             y=binary_values,
#             marker_color=binary_colors,
#             name='Binary'
#         ),
#         row=2, col=1
#     )
    
#     # 4. Diverging Data (Negative to Positive)
#     diverging_values = [-15, -5, 0, 8, 20]
#     diverging_categories = ['Item A', 'Item B', 'Item C', 'Item D', 'Item E']
    
#     fig.add_trace(
#         go.Bar(
#             x=diverging_categories,
#             y=diverging_values,
#             marker_color=[px.colors.diverging.RdBu[i] for i in [1, 3, 5, 7, 9]],
#             name='Diverging'
#         ),
#         row=2, col=2
#     )
    
#     # 5. Part-to-Whole Data (Proportions)
#     pie_values = [38, 27, 18, 10, 7]
#     pie_labels = ['Segment 1', 'Segment 2', 'Segment 3', 'Segment 4', 'Segment 5']
    
#     fig.add_trace(
#         go.Pie(
#             values=pie_values,
#             labels=pie_labels,
#             marker_colors=px.colors.qualitative.Pastel,
#             name='Parts to Whole',
#             textinfo='percent',
#             insidetextorientation='radial'
#         ),
#         row=3, col=1
#     )
    
#     # 6. Highlight and Context Data
#     highlight_x = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
#     highlight_y = [10, 12, 15, 22, 28, 32]
#     highlight_colors = ['#AEB6BF', '#AEB6BF', '#AEB6BF', '#AEB6BF', '#E74C3C', '#AEB6BF']
    
#     fig.add_trace(
#         go.Bar(
#             x=highlight_x,
#             y=highlight_y,
#             marker_color=highlight_colors,
#             name='Highlight'
#         ),
#         row=3, col=2
#     )
    
#     # Update layout
#     fig.update_layout(
#         height=900,
#         width=900,
#         title_text='Color Selection Guide Based on Data Types',
#         showlegend=False
#     )
    
#     fig.update_yaxes(title_text='Count', row=1, col=1)
#     fig.update_yaxes(title_text='Value', row=2, col=1)
#     fig.update_yaxes(title_text='Change', row=2, col=2)
#     fig.update_yaxes(title_text='Value', row=3, col=2)
    
#     # Ensure y-axis starts at 0 for binary data
#     fig.update_yaxes(range=[-0.1, 1.1], tickvals=[0, 1], ticktext=['No', 'Yes'], row=2, col=1)
    
#     # Add a reference line for diverging data
#     fig.add_shape(
#         type="line",
#         x0=-0.5, y0=0, x1=4.5, y1=0,
#         line=dict(color="black", width=1, dash="dash"),
#         row=2, col=2
#     )
    
#     # Add annotations explaining color choices
#     fig.add_annotation(
#         text="Use distinct hues<br>with similar brightness",
#         xref="x1", yref="y1",
#         x=2, y=48,
#         showarrow=False,
#         bgcolor="rgba(255,255,255,0.8)"
#     )
    
#     fig.add_annotation(
#         text="Use single-hue gradient<br>or multi-hue sequential",
#         xref="x2", yref="y2",
#         x=5, y=5,
#         showarrow=False,
#         bgcolor="rgba(255,255,255,0.8)"
#     )
    
#     fig.add_annotation(
#         text="Use contrasting colors<br>with clear meaning",
#         xref="x3", yref="y3",
#         x=2, y=0.5,
#         showarrow=False,
#         bgcolor="rgba(255,255,255,0.8)"
#     )
    
#     fig.add_annotation(
#         text="Use diverging scale<br>with neutral midpoint",
#         xref="x4", yref="y4",
#         x=2, y=-20,
#         showarrow=False,
#         bgcolor="rgba(255,255,255,0.8)"
#     )
    
#     fig.add_annotation(
#         text="Use related colors<br>that work together",
#         xref="x5", yref="y5",
#         x=0.5, y=0.5,
#         showarrow=False,
#         bgcolor="rgba(255,255,255,0.8)"
#     )
    
#     fig.add_annotation(
#         text="Use gray for context,<br>bright color for highlights",
#         xref="x6", yref="y6",
#         x=3, y=35,
#         showarrow=False,
#         bgcolor="rgba(255,255,255,0.8)"
#     )
    
#     fig.show()

# # Generate color selection guide
# color_selection_guide()

```
ValueError: Trace type 'pie' is not compatible with subplot type 'xy'
at grid position (3, 1)
See the docstring for the specs argument to plotly.subplots.make_subplots
for more information on subplot types
```

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Best practices for color usage in visualizations

# Create demonstration visualizations of color best practices
def color_best_practices():
    """Demonstrate best practices for color usage with examples"""
    
    # Set random seed for reproducibility
    np.random.seed(42)
    
    # 1. CONSISTENCY: Good vs. Bad Examples
    
    # Generate sample data for multiple charts
    categories = ['A', 'B', 'C', 'D', 'E']
    values1 = [23, 45, 56, 78, 42]
    values2 = [35, 67, 21, 49, 63]
    
    # Create subplots
    fig = make_subplots(rows=2, cols=2, 
                      subplot_titles=("Consistent Colors (Good)", "Inconsistent Colors (Bad)",
                                     "Restrained Color Usage (Good)", "Too Many Colors (Bad)"),
                      vertical_spacing=0.12,
                      horizontal_spacing=0.1)
    
    # Consistent color example (Good)
    consistent_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    
    fig.add_trace(
        go.Bar(x=categories, y=values1, name='Dataset 1', marker_color=consistent_colors),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(x=categories, y=values2, name='Dataset 2', mode='lines+markers',
                  marker=dict(color=consistent_colors)),
        row=1, col=1
    )
    
    # Inconsistent color example (Bad)
    fig.add_trace(
        go.Bar(x=categories, y=values1, name='Dataset 1', 
              marker_color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']),
        row=1, col=2
    )
    
    fig.add_trace(
        go.Scatter(x=categories, y=values2, name='Dataset 2', mode='lines+markers',
                  marker=dict(color=['#e377c2', '#8c564b', '#bcbd22', '#17becf', '#7f7f7f'])),
        row=1, col=2
    )
    
    # 2. RESTRAINT: Good vs. Bad Examples
    
    # Generate data for restrained example
    timeseries_data = np.cumsum(np.random.normal(0, 1, 50))
    threshold = 5
    
    # Restrained example (Good) - Limited colors with clear purpose
    x_data = list(range(len(timeseries_data)))
    
    fig.add_trace(
        go.Scatter(x=x_data, y=timeseries_data, name='Series', line=dict(color='#1f77b4')),
        row=2, col=1
    )
    
    fig.add_trace(
        go.Scatter(x=x_data, y=[threshold] * len(x_data), name='Threshold',
                  line=dict(color='#d62728', dash='dash')),
        row=2, col=1
    )
    
    # Areas above threshold - highlighting with color purpose
    above_threshold = [max(y-threshold, 0) for y in timeseries_data]
    x_above = [x for x, y in zip(x_data, timeseries_data) if y > threshold]
    y_above = [y for y in timeseries_data if y > threshold]
    
    fig.add_trace(
        go.Scatter(x=x_above, y=y_above, name='Above Threshold',
                  mode='markers', marker=dict(color='#d62728')),
        row=2, col=1
    )
    
    # Too many colors example (Bad) - Using a different color for each data point
    rainbow_colors = px.colors.sample_colorscale(
        px.colors.sequential.Turbo, [i/49 for i in range(50)]
    )
    
    fig.add_trace(
        go.Scatter(x=x_data, y=timeseries_data, name='Series',
                  mode='markers+lines', 
                  marker=dict(color=rainbow_colors, size=8),
                  line=dict(color='gray')),
        row=2, col=2
    )
    
    # Update layout
    fig.update_layout(
        height=800,
        width=900,
        title_text='Best Practices for Color Usage in Data Visualization',
        showlegend=False
    )
    
    fig.show()
    
    # 3. PURPOSE: Using color with clear purpose
    
    # Create a dataset for sales performance
    departments = ['Electronics', 'Clothing', 'Home Goods', 'Grocery', 'Beauty', 'Sports']
    target_values = [100, 85, 70, 110, 90, 65]
    actual_values = [95, 82, 60, 115, 95, 55]
    
    performance = []
    colors = []
    for target, actual in zip(target_values, actual_values):
        perf = (actual / target) * 100
        performance.append(perf)
        
        # Color based on performance
        if perf >= 100:  # Met or exceeded target
            colors.append('#2ecc71')  # Green
        elif perf >= 90:  # Close to target
            colors.append('#f39c12')  # Yellow/Orange
        else:  # Below target
            colors.append('#e74c3c')  # Red
    
    # Create comparison figure
    fig_purpose = make_subplots(rows=1, cols=2,
                             subplot_titles=("Purposeful Color (Performance Indicator)", 
                                          "Non-Purposeful Color (Decorative)"),
                             horizontal_spacing=0.15)
    
    # Purposeful color example
    fig_purpose.add_trace(
        go.Bar(
            x=departments,
            y=performance,
            marker_color=colors,
            text=[f"{p:.1f}%" for p in performance],
            textposition='outside',
            name='Performance'
        ),
        row=1, col=1
    )
    
    # Add a target line
    fig_purpose.add_trace(
        go.Scatter(
            x=departments,
            y=[100] * len(departments),
            line=dict(color='black', dash='dash', width=1),
            name='Target'
        ),
        row=1, col=1
    )
    
    # Non-purposeful color (decorative rainbow)
    fig_purpose.add_trace(
        go.Bar(
            x=departments,
            y=performance,
            marker_color=px.colors.qualitative.Plotly,  # Random decorative colors
            text=[f"{p:.1f}%" for p in performance],
            textposition='outside',
            name='Performance'
        ),
        row=1, col=2
    )
    
    # Add a target line
    fig_purpose.add_trace(
        go.Scatter(
            x=departments,
            y=[100] * len(departments),
            line=dict(color='black', dash='dash', width=1),
            name='Target'
        ),
        row=1, col=2
    )
    
    # Update layout
    fig_purpose.update_layout(
        height=400,
        width=900,
        title_text='Using Color with Clear Purpose',
        yaxis_title='% of Target',
        yaxis2_title='% of Target'
    )
    
    fig_purpose.update_yaxes(range=[50, 120], row=1, col=1)
    fig_purpose.update_yaxes(range=[50, 120], row=1, col=2)
    
    fig_purpose.show()

# Generate best practices examples
color_best_practices()

# Creating a guide for choosing colors based on data types
def color_selection_guide():
    """Create a guide for selecting colors based on data types"""
    
    # Generate sample data
    np.random.seed(42)
    
    # Create a figure with subplots for different data types
    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=(
            "Categorical Data (Qualitative)", 
            "Sequential Data (Low to High)",
            "Binary Data (Yes/No)", 
            "Diverging Data (Negative to Positive)",
            "Part-to-Whole Data (Proportions)",
            "Highlight and Context Data"
        ),
        vertical_spacing=0.1,
        horizontal_spacing=0.1
    )
    
    # 1. Categorical Data (Qualitative)
    categories = ['Category A', 'Category B', 'Category C', 'Category D', 'Category E']
    values = [45, 32, 38, 27, 42]
    
    fig.add_trace(
        go.Bar(
            x=categories,
            y=values,
            marker_color=px.colors.qualitative.Safe,
            name='Categories'
        ),
        row=1, col=1
    )
    
    # 2. Sequential Data (Low to High)
    x = np.linspace(0, 10, 20)
    y = np.linspace(0, 10, 20)
    X, Y = np.meshgrid(x, y)
    Z = np.sqrt(X**2 + Y**2)
    
    fig.add_trace(
        go.Heatmap(
            z=Z,
            colorscale='Viridis',
            showscale=False
        ),
        row=1, col=2
    )
    
    # 3. Binary Data (Yes/No)
    binary_categories = ['Group 1', 'Group 2', 'Group 3', 'Group 4', 'Group 5']
    binary_values = [1, 0, 1, 0, 1]  # 1 = Yes, 0 = No
    binary_colors = ['#2ecc71' if v == 1 else '#e74c3c' for v in binary_values]
    
    fig.add_trace(
        go.Bar(
            x=binary_categories,
            y=binary_values,
            marker_color=binary_colors,
            name='Binary'
        ),
        row=2, col=1
    )
    
    # 4. Diverging Data (Negative to Positive)
    diverging_values = [-15, -5, 0, 8, 20]
    diverging_categories = ['Item A', 'Item B', 'Item C', 'Item D', 'Item E']
    
    fig.add_trace(
        go.Bar(
            x=diverging_categories,
            y=diverging_values,
            marker_color=[px.colors.diverging.RdBu[i] for i in [1, 3, 5, 7, 9]],
            name='Diverging'
        ),
        row=2, col=2
    )
    
    # 5. Part-to-Whole Data (Proportions)
    pie_values = [38, 27, 18, 10, 7]
    pie_labels = ['Segment 1', 'Segment 2', 'Segment 3', 'Segment 4', 'Segment 5']
    
    fig_pie = go.Figure(go.Pie(
        values=pie_values,
        labels=pie_labels,
        marker_colors=px.colors.qualitative.Pastel,
        name='Parts to Whole',
        textinfo='percent',
        insidetextorientation='radial'
    ))
    
    fig_pie.update_layout(
        height=400,
        title_text="Part-to-Whole Visualization"
    )
    
    fig_pie.show()
    
    # 6. Highlight and Context Data
    highlight_x = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
    highlight_y = [10, 12, 15, 22, 28, 32]
    
    fig.add_trace(
        go.Scatter(
            x=highlight_x,
            y=highlight_y,
            mode='markers',
            marker=dict(color='#e74c3c', size=12),
            name='Highlight'
        ),
        row=3, col=2
    )
    
    fig.update_layout(
        height=900,
        width=900,
        title_text='Color Selection Guide for Different Data Types'
    )
    
    fig.show()

# Generate color selection guide examples
color_selection_guide()

## Color and Chart Types

Different chart types have distinct color requirements. Let's explore recommended color approaches for common visualization types:

### Recommended Color Approaches by Chart Type:

1. **Line Charts**
   - Multi-series: Qualitative palette with distinct colors
   - Single-series: Consider color to represent data meaning (e.g., red for negative trend)
   
2. **Scatter Plots**
   - Categories: Qualitative palette
   - Continuous third variable: Sequential or diverging color scale
   
3. **Bar/Column Charts**
   - Categorical: Qualitative palette
   - Sequential: Single-hue gradient
   - Highlighting specific bars: Neutral colors with accent highlights
   
4. **Heat Maps**
   - Correlation: Diverging palette with meaningful midpoint
   - Sequential data: Sequential palette
   
5. **Choropleth Maps**
   - Rate/ratio data: Sequential palette
   - Deviation data: Diverging palette
   - Categorical data: Qualitative palette
   
Let's demonstrate these principles with examples:

In [ ]:
# Color recommendations by chart type

# Function to demonstrate color usage in different chart types
def color_by_chart_type():
    # Set seed for reproducibility
    np.random.seed(42)
    
    # 1. LINE CHARTS
    # Create data for multi-series line chart
    x = list(range(10))
    y1 = np.cumsum(np.random.normal(0, 1, 10))
    y2 = np.cumsum(np.random.normal(0, 1, 10))
    y3 = np.cumsum(np.random.normal(0, 1, 10))
    y4 = np.cumsum(np.random.normal(0, 1, 10))
    
    # Create multi-series line chart with qualitative palette
    fig_lines = go.Figure()
    
    # Using a qualitative palette for multiple distinct series
    qual_colors = px.colors.qualitative.Safe
    
    fig_lines.add_trace(go.Scatter(x=x, y=y1, mode='lines+markers', name='Series A', 
                                 line=dict(color=qual_colors[0], width=2)))
    fig_lines.add_trace(go.Scatter(x=x, y=y2, mode='lines+markers', name='Series B', 
                                 line=dict(color=qual_colors[1], width=2)))
    fig_lines.add_trace(go.Scatter(x=x, y=y3, mode='lines+markers', name='Series C', 
                                 line=dict(color=qual_colors[2], width=2)))
    fig_lines.add_trace(go.Scatter(x=x, y=y4, mode='lines+markers', name='Series D', 
                                 line=dict(color=qual_colors[3], width=2)))
    
    fig_lines.update_layout(
        title='Line Chart: Use Qualitative Palette for Multiple Series',
        xaxis_title='Time Period',
        yaxis_title='Value',
        height=500,
        width=800,
        legend=dict(
            title="Series",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    fig_lines.show()
    
    # 2. SCATTER PLOTS
    # Create data for scatter plots with categories and continuous variables
    n = 80
    categories = np.random.choice(['Group A', 'Group B', 'Group C', 'Group D'], n)
    x = np.random.normal(0, 1, n)
    y = np.random.normal(0, 1, n)
    continuous_value = np.random.uniform(-2, 2, n)
    
    # Create scatter plots
    fig_scatter = make_subplots(rows=1, cols=2,
                             subplot_titles=("Categorical Groups: Use Qualitative Palette",
                                          "Continuous Variable: Use Sequential or Diverging"),
                             horizontal_spacing=0.14)
    
    # Categorical scatter plot
    fig_scatter.add_trace(
        go.Scatter(
            x=x, y=y, mode='markers',
            marker=dict(
                size=10,
                color=[{'Group A': 0, 'Group B': 1, 'Group C': 2, 'Group D': 3}[cat] for cat in categories],
                colorscale=px.colors.qualitative.G10,
                showscale=False
            ),
            text=categories,
            name='Categories',
            legendgroup='cat'
        ),
        row=1, col=1
    )
    
    # Add legend items manually
    for i, cat in enumerate(['Group A', 'Group B', 'Group C', 'Group D']):
        fig_scatter.add_trace(
            go.Scatter(
                x=[None], y=[None],
                mode='markers',
                marker=dict(size=10, color=px.colors.qualitative.G10[i]),
                name=cat,
                legendgroup='cat'
            ),
            row=1, col=1
        )
    
    # Continuous variable scatter plot
    fig_scatter.add_trace(
        go.Scatter(
            x=x, y=y, mode='markers',
            marker=dict(
                size=10,
                color=continuous_value,
                colorscale='RdBu_r',
                showscale=True,
                colorbar=dict(
                    title='Value',
                    thickness=15,
                    len=0.7,
                    y=0.5
                ),
                cmin=-2,
                cmid=0,
                cmax=2
            ),
            name='Continuous',
            showlegend=False
        ),
        row=1, col=2
    )
    
    fig_scatter.update_layout(
        height=500,
        width=900,
        title_text='Scatter Plot Color Recommendations',
    )
    
    fig_scatter.update_xaxes(title_text='X Value', row=1, col=1)
    fig_scatter.update_xaxes(title_text='X Value', row=1, col=2)
    fig_scatter.update_yaxes(title_text='Y Value', row=1, col=1)
    fig_scatter.update_yaxes(title_text='Y Value', row=1, col=2)
    
    fig_scatter.show()
    
    # 3. BAR/COLUMN CHARTS
    # Create data for different bar chart types
    categories = ['A', 'B', 'C', 'D', 'E', 'F']
    values = [23, 47, 32, 53, 28, 41]
    
    # Create bar charts with different color approaches
    fig_bars = make_subplots(rows=1, cols=3,
                          subplot_titles=("Categorical: Qualitative Palette", 
                                        "Sequential: Single-hue Gradient",
                                        "Highlighting Specific Bars"),
                          horizontal_spacing=0.1)
    
    # Categorical bar chart
    fig_bars.add_trace(
        go.Bar(
            x=categories,
            y=values,
            marker_color=px.colors.qualitative.Plotly,
            name='Categories'
        ),
        row=1, col=1
    )
    
    # Sequential bar chart
    # Sort values for sequential
    seq_values = sorted(values)
    seq_categories = [categories[values.index(v)] for v in seq_values]
    sequential_colors = px.colors.sequential.Blues[2:8]  # Take 6 colors from the Blues palette
    
    fig_bars.add_trace(
        go.Bar(
            x=seq_categories,
            y=seq_values,
            marker_color=sequential_colors,
            name='Sequential'
        ),
        row=1, col=2
    )
    
    # Highlighting specific bars
    highlight_colors = ['#AEB6BF'] * len(categories)  # Default gray
    highlight_colors[3] = '#E74C3C'  # Highlight the highest value in red
    
    fig_bars.add_trace(
        go.Bar(
            x=categories,
            y=values,
            marker_color=highlight_colors,
            name='Highlight'
        ),
        row=1, col=3
    )
    
    fig_bars.update_layout(
        height=400,
        width=1000,
        title_text='Bar Chart Color Recommendations',
        showlegend=False
    )
    
    fig_bars.update_yaxes(title_text='Value', row=1, col=1)
    fig_bars.update_yaxes(title_text='Value', row=1, col=2)
    fig_bars.update_yaxes(title_text='Value', row=1, col=3)
    
    fig_bars.show()
    
    # 4. HEAT MAPS
    # Create data for heat maps
    # Correlation matrix (diverging)
    corr_data = np.random.uniform(-0.7, 0.7, (6, 6))
    np.fill_diagonal(corr_data, 1)
    corr_data = (corr_data + corr_data.T) / 2  # Make it symmetric
    
    # Sequential data
    seq_data = np.random.randint(0, 100, (6, 6))
    
    # Create heat maps
    fig_heatmaps = make_subplots(rows=1, cols=2,
                               subplot_titles=("Correlation: Diverging Palette", 
                                             "Sequential Data: Sequential Palette"),
                               horizontal_spacing=0.14)
    
    # Correlation heatmap
    fig_heatmaps.add_trace(
        go.Heatmap(
            z=corr_data,
            x=['A', 'B', 'C', 'D', 'E', 'F'],
            y=['A', 'B', 'C', 'D', 'E', 'F'],
            colorscale='RdBu_r',
            zmid=0,
            colorbar=dict(
                title='Correlation',
                thickness=15,
                len=0.7,
                y=0.5,
                x=0.85
            )
        ),
        row=1, col=1
    )
    
    # Sequential heatmap
    fig_heatmaps.add_trace(
        go.Heatmap(
            z=seq_data,
            x=['A', 'B', 'C', 'D', 'E', 'F'],
            y=['1', '2', '3', '4', '5', '6'],
            colorscale='Viridis',
            colorbar=dict(
                title='Value',
                thickness=15,
                len=0.7,
                y=0.5,
                x=1.15
            )
        ),
        row=1, col=2
    )
    
    fig_heatmaps.update_layout(
        height=500,
        width=900,
        title_text='Heat Map Color Recommendations'
    )
    
    fig_heatmaps.show()
    
    # 5. CHOROPLETH MAPS
    # Create a simple US state-level dataset
    states = ['AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA', 
             'HI', 'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD',
             'MA', 'MI', 'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ',
             'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC',
             'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY']
    
    # Generate random data
    np.random.seed(42)
    # Sequential data (e.g., population density)
    sequential_values = np.random.uniform(10, 200, len(states))
    # Diverging data (e.g., election margin)
    diverging_values = np.random.uniform(-30, 30, len(states))
    # Categorical data (e.g., regions)
    categories = np.random.choice(['Northeast', 'Midwest', 'South', 'West'], len(states))
    
    # Create three choropleth maps
    fig_maps = make_subplots(rows=3, cols=1,
                           subplot_titles=("Sequential Data: Use Sequential Palette",
                                         "Diverging Data: Use Diverging Palette",
                                         "Categorical Data: Use Qualitative Palette"),
                           vertical_spacing=0.1,
                           specs=[[{"type": "choropleth"}],
                                 [{"type": "choropleth"}],
                                 [{"type": "choropleth"}]])
    
    # Sequential choropleth
    sequential_data = [dict(type='choropleth',
                          locations=states,
                          z=sequential_values,
                          locationmode='USA-states',
                          colorscale='YlOrRd',
                          colorbar=dict(
                              title='Value',
                              thickness=15,
                              len=0.4,
                              y=0.83
                          ))]
    
    fig_maps.add_trace(sequential_data[0], row=1, col=1)
    
    # Diverging choropleth
    diverging_data = [dict(type='choropleth',
                        locations=states,
                        z=diverging_values,
                        locationmode='USA-states',
                        colorscale='RdBu',
                        zmid=0,
                        colorbar=dict(
                            title='Value',
                            thickness=15,
                            len=0.4,
                            y=0.5
                        ))]
    
    fig_maps.add_trace(diverging_data[0], row=2, col=1)
    
    # Categorical choropleth
    # Create a numeric mapping for categories
    cat_map = {'Northeast': 0, 'Midwest': 1, 'South': 2, 'West': 3}
    cat_values = [cat_map[cat] for cat in categories]
    
    categorical_data = [dict(type='choropleth',
                          locations=states,
                          z=cat_values,
                          locationmode='USA-states',
                          colorscale=[
                              [0, px.colors.qualitative.Safe[0]],
                              [0.25, px.colors.qualitative.Safe[0]],
                              [0.25, px.colors.qualitative.Safe[1]],
                              [0.5, px.colors.qualitative.Safe[1]],
                              [0.5, px.colors.qualitative.Safe[2]],
                              [0.75, px.colors.qualitative.Safe[2]],
                              [0.75, px.colors.qualitative.Safe[3]],
                              [1, px.colors.qualitative.Safe[3]]
                          ],
                          colorbar=dict(
                              title='Region',
                              thickness=15,
                              tickvals=[0.125, 0.375, 0.625, 0.875],
                              ticktext=['Northeast', 'Midwest', 'South', 'West'],
                              len=0.4,
                              y=0.17
                          ))]
    
    fig_maps.add_trace(categorical_data[0], row=3, col=1)
    
    fig_maps.update_layout(
        height=900,
        width=800,
        title_text='Choropleth Map Color Recommendations',
        geo=dict(
            scope='usa',
            projection=dict(type='albers usa'),
            showlakes=True,
            lakecolor='rgb(255, 255, 255)'
        ),
        geo2=dict(
            scope='usa',
            projection=dict(type='albers usa'),
            showlakes=True,
            lakecolor='rgb(255, 255, 255)'
        ),
        geo3=dict(
            scope='usa',
            projection=dict(type='albers usa'),
            showlakes=True,
            lakecolor='rgb(255, 255, 255)'
        )
    )
    
    fig_maps.show()

# Generate chart type-specific color recommendations
color_by_chart_type()

## Summary of Color Selection Principles

Let's conclude with a comprehensive summary of the key principles for effective color selection in data visualizations:

### 1. Match Colors to Data Types
- **Categorical data**: Distinct hues with similar brightness (qualitative palettes)
- **Sequential data**: Single or multiple hues with varying lightness/saturation
- **Diverging data**: Two contrasting hues that diverge from a neutral midpoint
- **Binary data**: Two contrasting colors with clear meaning

### 2. Consider Visualization Context
- **Analytical context**: Use conservative, precise color schemes 
- **Presentation context**: Use more vibrant, memorable colors
- **Brand alignment**: Incorporate brand colors when appropriate
- **Cultural context**: Consider cultural color associations

### 3. Prioritize Accessibility
- Use colorblind-friendly palettes
- Don't rely solely on color to convey information
- Ensure sufficient contrast for text and important elements
- Test visualizations with color vision deficiency simulators

### 4. Apply Color Purposefully
- Use color to highlight the most important information
- Limit the number of colors to avoid overwhelming viewers
- Maintain consistency across related visualizations
- Choose colors that support the data's story, not distract from it

Let's create a final visualization that summarizes these principles: